# Revisão do M3

Este notebook verifica o M3 atual e orienta sua refatoração de acordo com a RQ vinculada no texto vigente do paper_v8.

**Conclusão principal:** o M3 pertence à RQ2 e não mede uso de IA. O indicador principal revisado é a proporção dos commits do repositório concentrada na fase final, definida como sete dias antes e sete dias depois do último voto T3 de cada equipe. A janela é dividida em pré e pós-apresentação; concentração de autoria e trajetória por corte permanecem como contexto.

## Estrutura recomendada

O M3 revisado é um conjunto pequeno de saídas complementares para a RQ2:

| Saída | Pergunta respondida | Status |
|---|---|---|
| **M3a — Participação dos commits na fase final** | Quando a atividade do repositório se concentra? | Resultado principal |
| **M3b — Concentração de autoria na fase final** | Quem concentra a atividade final? | Contexto de autoria |
| **M3c — Dinâmica da concentração e da atividade** | O padrão é estrutural ou surge perto da apresentação? | Contexto temporal |

Ordem de leitura:

1. vínculo com a RQ2 e auditoria do M3 legado;
2. definição comum de dados, âncora e janelas;
3. resultados M3a–M3c;
4. triagem de variações redundantes;
5. decisão de refatoração, mudanças no paper e pipeline.

## 1. Vínculo com a RQ2

O vínculo deve ser extraído do paper atual, não inferido pelo nome do arquivo.

A RQ esperada é:

> **RQ2:** How does the temporal density of repository activity contrast with the qualitative typification of human coordination friction across the project lifecycle?

O M3 aparece na subseção de métricas da RQ2. Na revisão, ele descreve densidade temporal e concentração de autoria; não mede fricção diretamente.

In [3]:
from hashlib import sha256
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().parent
PAPER_PATH = PROJECT_ROOT / "paper_v8/latex_code/main.tex"
GIT_COMMITS_PATH = PROJECT_ROOT / "data/lake/git_commits.parquet"
SOURCE_PATH = PROJECT_ROOT / "data/analysis/integration_friction_metrics.parquet"
M3_PATH = PROJECT_ROOT / "paper_v8/data/m3_author_concentration_density.csv"

paper_text = PAPER_PATH.read_text(encoding="utf-8")
rq_matches = dict(
    re.findall(
        r"\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n",
        paper_text,
    )
)
m3_position = paper_text.index(r"\textbf{M3 --")
rq2_section_position = paper_text.index(r"\subsubsection{RQ2:")
rq3_section_position = paper_text.index(r"\subsubsection{RQ3:")
detected_rq = "RQ2" if rq2_section_position < m3_position < rq3_section_position else "UNKNOWN"
assert rq_matches.get(detected_rq), "O texto da RQ vinculada não foi extraído"

paper_mapping_df = pd.DataFrame(
    [
        {
            "metric": "M3",
            "detected_rq": detected_rq,
            "rq_text": rq_matches[detected_rq],
            "paper_sha256": sha256(paper_text.encode("utf-8")).hexdigest(),
        }
    ]
)
paper_mapping_df

,metric,detected_rq,rq_text,paper_sha256
0,M3,RQ2,How does the temporal density of repository ac...,68bb2bbe3bd74611d7a2517844e943ca615845af21b8ec...


### 1.1 Configuração reproduzível

A configuração registra as decisões herdadas dos estudos M1/M2 e as específicas do M3. Ela evita misturar métricas, janelas, fontes e interpretações diferentes.

In [54]:
CONFIG = {
    "metric": "M3",
    "expected_rq": "RQ2",
    "analysis_grain": "team_semester_phase",
    "team_key": ["Semestre", "ID_Equipe"],
    "cuts": ["T1", "T2", "T3"],
    "expected_team_semesters": 14,
    "primary_metric": "final_commit_share",
    "secondary_metrics": [
        "final_phase_author_concentration",
        "structural_author_concentration",
        "rolling_author_concentration_7d",
    ],
    "primary_anchor": "last_t3_evaluator_vote",
    "window_days_before": 7,
    "window_days_after": 7,
    "rolling_window_days": 7,
    "rolling_step_days": 1,
    "rolling_end_day_range": [-21, 7],
    "denominator_end": "presentation_anchor_plus_7_days",
    "git_source": "read_only_parent_default_branch",
    "timestamp_field": "committer_date",
    "author_field": "author_email",
    "authorized_post_presentation_extensions": ["2025.2|TEAM_04"],
    "legacy_pre_t3_window_hours": 72,
    "no_activity_policy": "report_separately",
    "single_author_policy": "max_share_is_1_gini_is_0",
    "t3_is_final_deadline": False,
    "fork_created_at_is_primary_anchor": False,
    "llm_calls_required": False,
    "export_outputs": False,
    "regression_tolerance": 1e-12,
}

REQUIRED_CONFIG_FIELDS = {
    "metric",
    "expected_rq",
    "analysis_grain",
    "team_key",
    "cuts",
    "expected_team_semesters",
    "primary_metric",
    "secondary_metrics",
    "primary_anchor",
    "window_days_before",
    "window_days_after",
    "rolling_window_days",
    "rolling_step_days",
    "denominator_end",
    "git_source",
    "timestamp_field",
    "author_field",
    "authorized_post_presentation_extensions",
    "no_activity_policy",
    "llm_calls_required",
    "regression_tolerance",
}
missing_config = REQUIRED_CONFIG_FIELDS - set(CONFIG)
assert not missing_config, f"Configuração incompleta: {sorted(missing_config)}"
assert CONFIG["expected_rq"] == detected_rq, (
    f"M3 esperado em {CONFIG['expected_rq']}, encontrado em {detected_rq}"
)
CONFIG

{'metric': 'M3',
 'expected_rq': 'RQ2',
 'analysis_grain': 'team_semester_phase',
 'team_key': ['Semestre', 'ID_Equipe'],
 'cuts': ['T1', 'T2', 'T3'],
 'expected_team_semesters': 14,
 'primary_metric': 'final_commit_share',
 'secondary_metrics': ['final_phase_author_concentration',
  'structural_author_concentration',
  'rolling_author_concentration_7d'],
 'primary_anchor': 'last_t3_evaluator_vote',
 'window_days_before': 7,
 'window_days_after': 7,
 'rolling_window_days': 7,
 'rolling_step_days': 1,
 'rolling_end_day_range': [-21, 7],
 'denominator_end': 'presentation_anchor_plus_7_days',
 'git_source': 'read_only_parent_default_branch',
 'timestamp_field': 'committer_date',
 'author_field': 'author_email',
 'authorized_post_presentation_extensions': ['2025.2|TEAM_04'],
 'legacy_pre_t3_window_hours': 72,
 'no_activity_policy': 'report_separately',
 'single_author_policy': 'max_share_is_1_gini_is_0',
 't3_is_final_deadline': False,
 'fork_created_at_is_primary_anchor': False,
 'llm_cal

## 2. Auditoria do M3 legado

### 2.1 Proveniência

O fluxo legado é:

1. `git_commits.parquet`: commits com equipe, semestre, corte, autor local e churn;
2. `integration_friction_metrics.parquet`: concentrações por corte e na janela pré-T3 de 72 horas;
3. `m3_author_concentration_density.csv`: exportação publicada com T1, T2 e T3.

Não há classificação de commits como humanos ou produzidos por IA. O prefixo legado `ai_` não representa detecção de uso de IA.

In [5]:
git_commits_df = pd.read_parquet(GIT_COMMITS_PATH)
source_df = pd.read_parquet(SOURCE_PATH)
m3_original_df = pd.read_csv(M3_PATH, dtype={"Semestre": str})

for frame in (git_commits_df, source_df, m3_original_df):
    frame["Semestre"] = frame["Semestre"].astype(str)

TEAM_KEY = CONFIG["team_key"]
expected_keys = source_df[TEAM_KEY].drop_duplicates()

integrity_df = pd.DataFrame(
    [
        {"verificação": "14 equipes-semestre no artefato intermediário", "resultado": len(expected_keys) == 14},
        {"verificação": "Uma linha intermediária por equipe-semestre", "resultado": not source_df.duplicated(TEAM_KEY).any()},
        {"verificação": "42 linhas no M3 publicado", "resultado": len(m3_original_df) == 42},
        {"verificação": "Uma linha publicada por equipe-semestre-corte", "resultado": not m3_original_df.duplicated(TEAM_KEY + ["cut"]).any()},
        {"verificação": "Cortes publicados são T1, T2 e T3", "resultado": set(m3_original_df["cut"]) == set(CONFIG["cuts"])},
        {"verificação": "Todas as chaves publicadas existem na origem", "resultado": len(m3_original_df[TEAM_KEY].drop_duplicates().merge(expected_keys, on=TEAM_KEY)) == 14},
    ]
)

provenance_df = pd.DataFrame(
    [
        {"artefato": "Commits", "arquivo": "data/lake/git_commits.parquet", "linhas": len(git_commits_df), "unidade": "commit"},
        {"artefato": "Métricas intermediárias", "arquivo": "data/analysis/integration_friction_metrics.parquet", "linhas": len(source_df), "unidade": "equipe-semestre"},
        {"artefato": "M3 publicado", "arquivo": "paper_v8/data/m3_author_concentration_density.csv", "linhas": len(m3_original_df), "unidade": "equipe-semestre-corte"},
    ]
)

display(provenance_df)
integrity_df

,artefato,arquivo,linhas,unidade
0,Commits,data/lake/git_commits.parquet,812,commit
1,Métricas intermediárias,data/analysis/integration_friction_metrics.par...,14,equipe-semestre
2,M3 publicado,paper_v8/data/m3_author_concentration_density.csv,42,equipe-semestre-corte


,verificação,resultado
0,14 equipes-semestre no artefato intermediário,True
1,Uma linha intermediária por equipe-semestre,True
2,42 linhas no M3 publicado,True
3,Uma linha publicada por equipe-semestre-corte,True
4,"Cortes publicados são T1, T2 e T3",True
5,Todas as chaves publicadas existem na origem,True


### 2.2 Validação automática M3 → RQ2

A execução deve parar se:

- o M3 não estiver dentro da subseção RQ2;
- o texto da RQ2 não for encontrado;
- o paper passar a vincular M3 a outra RQ.

O trecho abaixo registra a evidência textual usada na validação.

In [6]:
m3_section_excerpt = paper_text[m3_position:m3_position + 900]
validation_log = {
    "metric": "M3",
    "expected_rq": CONFIG["expected_rq"],
    "detected_rq": detected_rq,
    "rq_text": rq_matches[detected_rq],
    "m3_section_excerpt": m3_section_excerpt,
    "status": "pass" if detected_rq == CONFIG["expected_rq"] else "fail",
}
assert validation_log["status"] == "pass", validation_log
validation_log

{'metric': 'M3',
 'expected_rq': 'RQ2',
 'detected_rq': 'RQ2',
 'rq_text': 'How does the temporal density of repository activity contrast with the qualitative typification of human coordination friction across the project lifecycle?',
 'm3_section_excerpt': "\\textbf{M3 -- Pre-Deadline Author-Concentration Density.} For team $i$ in\nsemester $s$, we take every commit in the 72-hour window immediately\npreceding that team's $T_3$ checkpoint and group it by the committing\nteammate's identifier. Let $C(a,i,s)$ be teammate $a$'s commit count in that\nwindow and $C(i,s)$ the window's total commit count; we report the median\nper-teammate commit share and the Gini coefficient of the per-teammate\ncommit-count distribution:\n\\[\n\\begin{aligned}\n\\mathrm{AuthorShare}(i,s)\n  &= \\underset{a}{\\mathrm{median}}\n     \\left(\\frac{C(a,i,s)}{C(i,s)}\\right), \\\\\n\\mathrm{AuthorGini}(i,s)\n  &= \\mathrm{Gini}\\bigl(\\{C(a,i,s)\\}_a\\bigr).\n\\end{aligned}\n\\]\nWe emphasize what this metric 

### 2.3 Correção aritmética e inconsistência conceitual

Há duas respostas:

- **Sim**, o CSV publicado reproduz corretamente a concentração calculada nos cortes completos T1, T2 e T3.
- **Não**, esses valores não correspondem à definição textual de janela fixa de 72 horas antes de T3.

A auditoria abaixo recalcula o CSV diretamente dos commits e constrói separadamente a métrica de 72 horas disponível no artefato intermediário.

In [7]:
def gini(values: pd.Series) -> float:
    ordered = np.sort(values.astype(float).to_numpy())
    if len(ordered) <= 1 or ordered.sum() == 0:
        return 0.0
    index = np.arange(1, len(ordered) + 1)
    return float(
        2 * np.sum(index * ordered) / (len(ordered) * ordered.sum())
        - (len(ordered) + 1) / len(ordered)
    )


def calculate_cut_metrics(
    commits: pd.DataFrame,
    team_keys: pd.DataFrame,
    cuts: list[str],
) -> pd.DataFrame:
    rows = []
    for key in team_keys.to_dict("records"):
        team_commits = commits[
            commits["Semestre"].eq(key["Semestre"])
            & commits["ID_Equipe"].eq(key["ID_Equipe"])
        ]
        for cut in cuts:
            current = team_commits[team_commits["temporal_marker"].eq(cut)]
            counts = current[CONFIG["author_field"]].dropna().value_counts()
            shares = counts / len(current) if len(current) else pd.Series(dtype=float)
            rows.append(
                {
                    **key,
                    "cut": cut,
                    "author_n": int(len(counts)),
                    "author_share_median": float(shares.median()) if len(shares) else np.nan,
                    "max_author_share": float(shares.max()) if len(shares) else np.nan,
                    "commit_n": int(len(current)),
                    "churn": float((current["lines_added"] + current["lines_deleted"]).sum()),
                    "author_gini": gini(counts) if len(counts) else np.nan,
                }
            )
    return pd.DataFrame(rows)


recalculated_cut_df = calculate_cut_metrics(
    git_commits_df,
    expected_keys,
    CONFIG["cuts"],
)

COMPARE_COLUMNS = ["author_share_median", "commit_n", "churn", "author_gini"]
cut_comparison_df = recalculated_cut_df.merge(
    m3_original_df,
    on=TEAM_KEY + ["cut"],
    suffixes=("_recalculado", "_publicado"),
    validate="one_to_one",
)
max_errors = {
    column: (
        cut_comparison_df[f"{column}_recalculado"]
        - cut_comparison_df[f"{column}_publicado"]
    ).abs().max()
    for column in COMPARE_COLUMNS
}

pre_t3_72h_df = source_df[
    TEAM_KEY
    + [
        "ai_commit_n_before_t3_window",
        "ai_churn_before_t3_window",
        "ai_author_n_before_t3_window",
        "ai_max_author_share_before_t3_window",
        "ai_author_share_median_before_t3_window",
        "ai_author_share_iqr_before_t3_window",
        "ai_gini_before_t3_window",
        "ai_t3_window_hours",
    ]
].rename(
    columns={
        "ai_commit_n_before_t3_window": "commit_n",
        "ai_churn_before_t3_window": "churn",
        "ai_author_n_before_t3_window": "author_n",
        "ai_max_author_share_before_t3_window": "max_author_share",
        "ai_author_share_median_before_t3_window": "author_share_median",
        "ai_author_share_iqr_before_t3_window": "author_share_iqr",
        "ai_gini_before_t3_window": "author_gini",
        "ai_t3_window_hours": "window_hours",
    }
)

calculation_audit_df = pd.DataFrame(
    [
        {"verificação": "CSV confere com recálculo por cortes completos", "resultado": max(max_errors.values()) < CONFIG["regression_tolerance"]},
        {"verificação": "Janela pré-T3 configurada em 72 horas", "resultado": pre_t3_72h_df["window_hours"].eq(72).all()},
        {"verificação": "CSV publicado contém a janela pré-T3 de 72h", "resultado": False},
    ]
)

display(calculation_audit_df)
pd.Series(max_errors, name="erro máximo")

,verificação,resultado
0,CSV confere com recálculo por cortes completos,True
1,Janela pré-T3 configurada em 72 horas,True
2,CSV publicado contém a janela pré-T3 de 72h,False


author_share_median    8.326673e-17
commit_n               0.000000e+00
churn                  0.000000e+00
author_gini            6.938894e-17
Name: erro máximo, dtype: float64

### 2.4 Limites para responder à RQ2

O M3 legado responde **parcialmente** à RQ2.

- A densidade temporal da atividade é medida diretamente pelo M4.
- A fricção qualitativa é medida pelo M5.
- O M3 acrescenta contexto sobre quando os commits ocorrem e como se distribuem entre autores.

M3 não mede coordenação, integração, esforço, produtividade ou uso de IA. A ausência de commits deve ser reportada separadamente; sem atividade, não existe concentração de autoria observável.

In [8]:
def summarize_concentration(
    frame: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    active = frame[frame["commit_n"].gt(0)].copy()
    totals = frame.groupby(group_columns, as_index=False).agg(
        equipes=("ID_Equipe", "size"),
        equipes_ativas=("commit_n", lambda values: values.gt(0).sum()),
        commits=("commit_n", "sum"),
        churn=("churn", "sum"),
    )
    concentration = active.groupby(group_columns, as_index=False).agg(
        mediana_autores_ativos=("author_n", "median"),
        mediana_maior_participacao=("max_author_share", "median"),
        mediana_participacao_por_autor=("author_share_median", "median"),
        mediana_gini=("author_gini", "median"),
    )
    return totals.merge(concentration, on=group_columns, how="left", validate="one_to_one")


cut_summary_df = summarize_concentration(
    recalculated_cut_df,
    ["Semestre", "cut"],
)

window_summary_by_semester_df = summarize_concentration(
    pre_t3_72h_df,
    ["Semestre"],
)
window_summary_pooled_df = pd.DataFrame(
    [
        {
            "Semestre": "Todos",
            "equipes": len(pre_t3_72h_df),
            "equipes_ativas": int(pre_t3_72h_df["commit_n"].gt(0).sum()),
            "commits": int(pre_t3_72h_df["commit_n"].sum()),
            "churn": float(pre_t3_72h_df["churn"].sum()),
            "mediana_autores_ativos": pre_t3_72h_df.loc[pre_t3_72h_df["commit_n"].gt(0), "author_n"].median(),
            "mediana_maior_participacao": pre_t3_72h_df["max_author_share"].median(),
            "mediana_participacao_por_autor": pre_t3_72h_df["author_share_median"].median(),
            "mediana_gini": pre_t3_72h_df["author_gini"].median(),
        }
    ]
)
window_summary_df = pd.concat(
    [window_summary_by_semester_df, window_summary_pooled_df],
    ignore_index=True,
)

display(cut_summary_df.round(3))
window_summary_df.round(3)

,Semestre,cut,equipes,equipes_ativas,commits,churn,mediana_autores_ativos,mediana_maior_participacao,mediana_participacao_por_autor,mediana_gini
0,2025.2,T1,9,4,24,18814.0,2.0,0.619,0.500,0.071
1,2025.2,T2,9,6,141,206283.0,5.5,0.475,0.124,0.404
2,2025.2,T3,9,9,352,13372689.0,3.0,0.500,0.333,0.190
3,2026.1,T1,5,5,87,31800.0,3.0,0.636,0.273,0.364
4,2026.1,T2,5,4,88,36116.0,6.5,0.393,0.128,0.375
5,2026.1,T3,5,5,120,129442.0,4.0,0.447,0.265,0.221


,Semestre,equipes,equipes_ativas,commits,churn,mediana_autores_ativos,mediana_maior_participacao,mediana_participacao_por_autor,mediana_gini
0,2025.2,9,3,111,37247.0,4.0,0.511,0.178,0.483
1,2026.1,5,4,32,34861.0,3.0,0.536,0.293,0.298
2,Todos,14,7,143,72108.0,4.0,0.511,0.265,0.311


### 2.5 Medidas legadas de concentração

Nenhuma medida isolada é suficiente:

- **autores ativos:** cobertura da colaboração observada;
- **maior participação de um autor:** dominância direta;
- **Gini entre autores ativos:** desigualdade entre quem contribuiu;
- **mediana das participações:** fortemente determinada pelo número de autores.

Com um único autor ativo, a maior participação é 1 e o Gini é 0. Há concentração máxima em uma pessoa, mas nenhuma desigualdade dentro de um conjunto unitário.

A revisão usa `max_author_share` como medida principal de autoria, sempre acompanhada de `author_n` e `commit_n`. O Gini permanece complementar.

In [10]:
import plotly.graph_objects as go

concentration_figure = go.Figure()
for semester, group in cut_summary_df.groupby("Semestre", sort=False):
    concentration_figure.add_trace(
        go.Scatter(
            x=group["cut"],
            y=group["mediana_maior_participacao"],
            mode="lines+markers",
            name=semester,
        )
    )

concentration_figure.update_layout(
    title="Concentração de autoria entre equipes com atividade",
    xaxis_title="Corte",
    yaxis_title="Mediana da maior participação",
    yaxis_range=[0, 1],
    legend_title_text="Semestre",
)
display(HTML(concentration_figure.to_html(include_plotlyjs="cdn", full_html=False)))

### 2.6 Testes exploratórios do legado

A RQ2 é descritiva. M3 pode verificar se maior volume coincide com maior concentração, mas não pode testar fricção diretamente:

- M3 está no nível equipe-semestre-corte;
- M5 possui apenas um valor agregado por corte;
- os três valores atuais de M5 são iguais a 8/10.

Sem variação e sem correspondência de granularidade, uma correlação M3–M5 não é válida. As associações abaixo entre concentração, commits e churn são exploratórias e não causais.

In [11]:
from scipy.stats import spearmanr


def spearman_table(frame: pd.DataFrame, window: str) -> pd.DataFrame:
    rows = []
    active = frame[frame["commit_n"].gt(0)].copy()
    groups = active.groupby("cut") if "cut" in active.columns else [("pre-T3 72h", active)]
    for cut, group in groups:
        for concentration in ("max_author_share", "author_gini"):
            for activity in ("commit_n", "churn"):
                valid = group[[concentration, activity]].dropna()
                rho, p_value = spearmanr(valid[concentration], valid[activity])
                rows.append(
                    {
                        "janela": window,
                        "corte": cut,
                        "concentracao": concentration,
                        "atividade": activity,
                        "n": len(valid),
                        "rho": rho,
                        "p_value": p_value,
                    }
                )
    return pd.DataFrame(rows)


association_df = pd.concat(
    [
        spearman_table(recalculated_cut_df, "corte completo"),
        spearman_table(pre_t3_72h_df, "janela fixa"),
    ],
    ignore_index=True,
)
association_df.round(3)

,janela,corte,concentracao,atividade,n,rho,p_value
0,corte completo,T1,max_author_share,commit_n,9,-0.729,0.026
1,corte completo,T1,max_author_share,churn,9,-0.924,0.000
2,corte completo,T1,author_gini,commit_n,9,0.831,0.006
3,corte completo,T1,author_gini,churn,9,0.487,0.183
4,corte completo,T2,max_author_share,commit_n,10,-0.436,0.208
5,corte completo,T2,max_author_share,churn,10,0.839,0.002
6,corte completo,T2,author_gini,commit_n,10,0.869,0.001
7,corte completo,T2,author_gini,churn,10,-0.042,0.907
8,corte completo,T3,max_author_share,commit_n,14,-0.164,0.576
9,corte completo,T3,max_author_share,churn,14,0.427,0.128


In [12]:
full_t3 = recalculated_cut_df[
    recalculated_cut_df["cut"].eq("T3") & recalculated_cut_df["commit_n"].gt(0)
]
active_72h = pre_t3_72h_df[pre_t3_72h_df["commit_n"].gt(0)]

baseline_comparison_df = pd.DataFrame(
    [
        {
            "definição": "T3 completo publicado",
            "equipes_ativas": len(full_t3),
            "mediana_author_share": full_t3["author_share_median"].median(),
            "mediana_max_share": full_t3["max_author_share"].median(),
            "mediana_gini": full_t3["author_gini"].median(),
        },
        {
            "definição": "72h antes de T3",
            "equipes_ativas": len(active_72h),
            "mediana_author_share": active_72h["author_share_median"].median(),
            "mediana_max_share": active_72h["max_author_share"].median(),
            "mediana_gini": active_72h["author_gini"].median(),
        },
    ]
)
baseline_comparison_df.round(3)

,definição,equipes_ativas,mediana_author_share,mediana_max_share,mediana_gini
0,T3 completo publicado,14,0.299,0.500,0.206
1,72h antes de T3,7,0.265,0.511,0.311


### Interpretação dos testes exploratórios

Aqui, “equipe ativa” significa apenas **equipe com pelo menos um commit nas 72 horas imediatamente anteriores ao marco T3**. Não significa que a equipe esteve inativa no projeto ou no corte T3 completo.

Somente 7 das 14 equipes possuem commits no intervalo $[T3-72h, T3)$:

- 2025.2: TEAM_02, TEAM_04 e TEAM_09 (3/9);
- 2026.1: TEAM_02, TEAM_03, TEAM_04 e TEAM_05 (4/5).

As outras sete equipes possuem zero commits nessa janela específica. Como não há commits, também não há distribuição de autoria a partir da qual calcular concentração. Por isso, elas permanecem no denominador de cobertura, mas são excluídas dos resumos de concentração da janela de 72 horas.

Os resultados dessa janela não mostram associação estatisticamente clara entre concentração e volume, com apenas sete equipes com commits.

As associações por corte não devem ser tratadas como hipóteses independentes confirmadas:

- foram calculadas em amostras pequenas;
- existem múltiplas comparações;
- número de commits, número de autores, maior participação e Gini possuem relações matemáticas entre si;
- churn muito alto pode ser dominado por arquivos gerados ou eventos extremos.

Para a RQ2, esses resultados são contexto descritivo. O contraste principal continua sendo M4 versus M5.

## 3. Âncora temporal e fonte Git

### 3.1 Por que o início de T3 não é o deadline

A ausência de commits antes de T3 não implica bom planejamento. Pode representar estabilidade, conclusão antecipada, trabalho fora do Git, atraso ou baixa produção.

A âncora legada é o início do intervalo T3, não a apresentação final de cada equipe:

- em 2025.2, T3 cobre duas sessões, 05/12 e 12/12;
- em 2026.1, T3 ocorre em 19/06;
- as apresentações finais ocorreram depois de T3.

Logo, $[T3-72h,T3)$ não representa as últimas 72 horas do projeto. A revisão usa o último voto T3 de cada equipe como proxy do fim da sessão e o histórico read-only do repositório original.

In [20]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import EVALUATOR_TEMPORAL_CUTS

WINDOW_DAYS = [1, 3, 7, 14, 21, 28]


def calculate_window_metrics(
    commits: pd.DataFrame,
    team_keys: pd.DataFrame,
    window_days: list[int],
) -> pd.DataFrame:
    rows = []
    for key in team_keys.to_dict("records"):
        t3_start = pd.Timestamp(
            EVALUATOR_TEMPORAL_CUTS[key["Semestre"]]["T3"][0],
            tz="UTC",
        )
        t3_end_exclusive = pd.Timestamp(
            EVALUATOR_TEMPORAL_CUTS[key["Semestre"]]["T3"][1],
            tz="UTC",
        ) + pd.Timedelta(days=1)
        team = commits[
            commits["Semestre"].eq(key["Semestre"])
            & commits["ID_Equipe"].eq(key["ID_Equipe"])
        ]
        for days in window_days:
            window_start = t3_start - pd.Timedelta(days=days)
            window = team[
                team["timestamp"].ge(window_start)
                & team["timestamp"].lt(t3_start)
            ]
            counts = window[CONFIG["author_field"]].dropna().value_counts()
            shares = counts / len(window) if len(window) else pd.Series(dtype=float)
            rows.append(
                {
                    **key,
                    "window_days": days,
                    "t3_interval_start": t3_start,
                    "t3_interval_end_exclusive": t3_end_exclusive,
                    "commit_n": int(len(window)),
                    "author_n": int(len(counts)),
                    "max_author_share": float(shares.max()) if len(shares) else np.nan,
                    "author_gini": gini(counts) if len(counts) else np.nan,
                    "commits_before_window": int(team["timestamp"].lt(window_start).sum()),
                    "commits_during_t3_interval": int(
                        (team["timestamp"].ge(t3_start) & team["timestamp"].lt(t3_end_exclusive)).sum()
                    ),
                    "commits_after_t3_interval": int(team["timestamp"].ge(t3_end_exclusive).sum()),
                }
            )
    return pd.DataFrame(rows)


window_sensitivity_df = calculate_window_metrics(
    git_commits_df,
    expected_keys,
    WINDOW_DAYS,
)
window_sensitivity_summary_df = (
    window_sensitivity_df.groupby("window_days", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        equipes_com_commits=("commit_n", lambda values: values.gt(0).sum()),
        commits=("commit_n", "sum"),
        mediana_autores=("author_n", lambda values: values[values.gt(0)].median()),
        mediana_max_share=("max_author_share", "median"),
        mediana_gini=("author_gini", "median"),
    )
)
window_sensitivity_summary_df["cobertura"] = (
    window_sensitivity_summary_df["equipes_com_commits"]
    / window_sensitivity_summary_df["equipes"]
)
window_sensitivity_summary_df.round(3)

,window_days,equipes,equipes_com_commits,commits,mediana_autores,mediana_max_share,mediana_gini,cobertura
0,1,14,7,69,3.0,0.562,0.167,0.500
1,3,14,7,143,4.0,0.511,0.311,0.500
2,7,14,8,184,4.5,0.506,0.323,0.571
3,14,14,9,226,4.0,0.500,0.250,0.643
4,21,14,11,335,4.0,0.500,0.303,0.786
5,28,14,12,403,3.5,0.544,0.404,0.857


In [21]:
TEAM_SIGNALS_PATH = (
    PROJECT_ROOT / "paper_v4/advanced_metrics/outputs/team_level_signals.csv"
)
team_signals_df = pd.read_csv(TEAM_SIGNALS_PATH, dtype={"Semestre": str})

window_72h_context_df = (
    window_sensitivity_df[window_sensitivity_df["window_days"].eq(3)]
    .assign(has_commits_72h=lambda frame: frame["commit_n"].gt(0))
    .merge(team_signals_df, on=TEAM_KEY, validate="one_to_one")
)

planning_hypothesis_df = (
    window_72h_context_df.groupby("has_commits_72h", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        planejamento_observado_n=("t1_planning_score", "count"),
        planejamento_mediano=("t1_planning_score", "median"),
        commits_anteriores_mediana=("commits_before_window", "median"),
        commits_durante_intervalo_T3_mediana=("commits_during_t3_interval", "median"),
        commits_apos_intervalo_T3_mediana=("commits_after_t3_interval", "median"),
        rework_mediano=("rework_churn_t3", "median"),
        desenvolvimento_adiado_mediano=("deferred_churn_t3", "median"),
        escopo_mediano=("scope_applicability_mean", "median"),
        complexidade_mediana=("technical_complexity_mean", "median"),
    )
)
planning_hypothesis_df["grupo"] = planning_hypothesis_df[
    "has_commits_72h"
].map(
    {
        False: "Sem commits nas 72h antes do início de T3",
        True: "Com commits nas 72h antes do início de T3",
    }
)
planning_hypothesis_df.drop(columns="has_commits_72h").round(3)

,equipes,planejamento_observado_n,planejamento_mediano,commits_anteriores_mediana,commits_durante_intervalo_T3_mediana,commits_apos_intervalo_T3_mediana,rework_mediano,desenvolvimento_adiado_mediano,escopo_mediano,complexidade_mediana,grupo
0,7,3,3.0,8.0,6.0,0.0,172.0,2532.0,1.917,1.778,Sem commits nas 72h antes do início de T3
1,7,6,3.5,55.0,8.0,0.0,4459.0,20005.0,1.889,1.778,Com commits nas 72h antes do início de T3


In [22]:
zero_72h_teams_df = window_72h_context_df[
    ~window_72h_context_df["has_commits_72h"]
][
    TEAM_KEY
    + [
        "commits_before_window",
        "commits_during_t3_interval",
        "commits_after_t3_interval",
        "t1_planning_score",
        "rework_churn_t3",
        "deferred_churn_t3",
    ]
].copy()

zero_72h_teams_df["perfil_temporal"] = np.select(
    [
        zero_72h_teams_df["commits_before_window"].eq(0),
        zero_72h_teams_df["commits_during_t3_interval"].eq(0),
    ],
    [
        "Sem commits antes da janela; atividade somente no intervalo T3",
        "Atividade anterior; sem commits no intervalo T3",
    ],
    default="Atividade anterior e também no intervalo T3",
)
zero_72h_teams_df.sort_values(TEAM_KEY)

,Semestre,ID_Equipe,commits_before_window,commits_during_t3_interval,commits_after_t3_interval,t1_planning_score,rework_churn_t3,deferred_churn_t3,perfil_temporal
0,2025.2,TEAM_01,8,11,0,3.0,1154.0,1452.0,Atividade anterior e também no intervalo T3
4,2025.2,TEAM_03,0,9,0,NaN,0.0,16418.0,Sem commits antes da janela; atividade somente...
8,2025.2,TEAM_05,0,1,0,NaN,0.0,2532.0,Sem commits antes da janela; atividade somente...
10,2025.2,TEAM_07,67,6,0,5.0,59.0,160240.0,Atividade anterior e também no intervalo T3
11,2025.2,TEAM_08,25,1,0,NaN,172.0,0.0,Atividade anterior e também no intervalo T3
13,2025.2,TEAM_10,12,6,0,NaN,1041.0,1161.0,Atividade anterior e também no intervalo T3
1,2026.1,TEAM_01,2,1,0,1.0,2780.0,4431.0,Atividade anterior e também no intervalo T3


### 3.2 Cobertura Git após o intervalo T3

No dataset legado, todos os commits posteriores ao início de T3 ocorrem dentro do intervalo configurado para T3. Não há commits após o fim desse intervalo.

Isso não prova ausência de atividade até a apresentação final: os forks centrais não foram sincronizados uniformemente. A revisão resolve o problema por mirrors read-only dos repositórios originais.

In [23]:
post_t3_availability_df = (
    window_sensitivity_df[window_sensitivity_df["window_days"].eq(3)]
    .groupby("Semestre", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        commits_durante_intervalo_T3=("commits_during_t3_interval", "sum"),
        equipes_com_commits_durante_T3=(
            "commits_during_t3_interval",
            lambda values: values.gt(0).sum(),
        ),
        commits_apos_intervalo_T3=("commits_after_t3_interval", "sum"),
        equipes_com_commits_apos_T3=(
            "commits_after_t3_interval",
            lambda values: values.gt(0).sum(),
        ),
    )
)
post_t3_availability_df

,Semestre,equipes,commits_durante_intervalo_T3,equipes_com_commits_durante_T3,commits_apos_intervalo_T3,equipes_com_commits_apos_T3
0,2025.2,9,95,9,0,0
1,2026.1,5,29,5,0,0


### 3.3 Fork como evidência auxiliar

O timestamp do fork reproduz as sessões, mas não é a âncora principal porque alguns forks foram criados tardiamente:

- 2025.2: cinco forks em 05/12 e quatro em 12/12;
- 2026.1: cinco forks em 22/06.

`created_at` e `pushed_at` permanecem úteis para auditoria. TEAM_04/2025.2 teve atualização posterior ao fork, confirmando que o estado atual do fork não é necessariamente o snapshot da apresentação.

A âncora principal passa a ser o último voto T3. O fork fica apenas como evidência auxiliar de sessão e integridade.

In [39]:
import subprocess

# Metadados públicos capturados em 2026-09-24. A análise principal não depende
# destes timestamps; eles permanecem somente para auditorar o uso anterior do fork.
FORK_METADATA_RECORDS = [
    ("extensao3-2025_2-team_01-software_development_extension_lll-1930641bc289", "2025-12-12T13:03:45Z", "2025-12-12T03:29:35Z", "Sergiosampjr/software_development_extension_lll"),
    ("extensao3-2025_2-team_02-EstudaAI-1953213cc45b", "2025-12-05T12:24:18Z", "2025-12-05T04:37:37Z", "HChuvas/EstudaAI"),
    ("extensao3-2025_2-team_03-career-path-ai-671a7f548397", "2025-12-12T12:12:49Z", "2025-12-12T11:43:42Z", "daviolvr/career-path-ai"),
    ("extensao3-2025_2-team_04-Bulicoso-1596a9c51e60", "2025-12-05T12:10:31Z", "2025-12-12T13:21:41Z", "SilvioGoncalvesXJr/Bulicoso"),
    ("extensao3-2025_2-team_05-ecoroteiro-ex3-829bfc0e2da3", "2025-12-12T13:01:43Z", "2025-12-12T02:47:21Z", "jamileehellen/ecoroteiro-ex3"),
    ("extensao3-2025_2-team_07-Projeto-Research-Flow-52c168a459d8", "2025-12-12T11:41:27Z", "2025-12-11T13:31:01Z", "GlaucoCiprianoMoreira/Projeto-Research-Flow"),
    ("extensao3-2025_2-team_08-ExamForge-80e19be1a764", "2025-12-05T12:11:42Z", "2025-12-05T00:55:07Z", "Gabriel-marques-araujo/ExamForge"),
    ("extensao3-2025_2-team_09-SoFiA-59690ae48d74", "2025-12-05T12:55:47Z", "2025-12-05T12:43:11Z", "suyanecarvalho/SoFiA"),
    ("extensao3-2025_2-team_10-Extensao-3-Synapse-1ea204dc650c", "2025-12-05T13:19:58Z", "2025-12-05T12:21:49Z", "ianjsm/Extensao-3-Synapse"),
    ("extensao3-2026_1-team1-foodguard-4f38369906d5", "2026-06-22T09:21:26Z", "2026-06-19T10:39:18Z", "VictorManoel-Timbo/extensao-3-projeto"),
    ("extensao3-2026_1-team2-revisai-fc0269144b84", "2026-06-22T09:22:36Z", "2026-06-19T10:09:08Z", "Kayquemts/RevisAI"),
    ("extensao3-2026_1-team3-interview-17c184cfaa5e", "2026-06-22T09:23:34Z", "2026-06-19T13:46:37Z", "Emanoel-de-Moura-Silva/Extens-o-III"),
    ("extensao3-2026_1-team4-SmartFlow-WhatsApp-AI-2d2e1132b2a4", "2026-06-22T09:24:41Z", "2026-06-19T18:14:22Z", "gabrielalbuq/SmartFlow-WhatsApp-AI"),
    ("extensao3-2026_1-team5-enembot-836dc10bc759", "2026-06-22T09:25:36Z", "2026-06-19T15:43:05Z", "RenannLimaa/extensaoIII"),
]

fork_deadline_metadata_df = pd.DataFrame(
    FORK_METADATA_RECORDS,
    columns=[
        "repository",
        "fork_created_at",
        "current_pushed_at",
        "parent_repository",
    ],
)
fork_deadline_metadata_df["fork_created_at"] = pd.to_datetime(
    fork_deadline_metadata_df["fork_created_at"],
    utc=True,
)
fork_deadline_metadata_df["current_pushed_at"] = pd.to_datetime(
    fork_deadline_metadata_df["current_pushed_at"],
    utc=True,
)
fork_deadline_metadata_df["is_fork"] = True

assert len(fork_deadline_metadata_df) == 14
assert fork_deadline_metadata_df["fork_created_at"].notna().all()
fork_deadline_metadata_df.sort_values("fork_created_at")

,repository,fork_created_at,current_pushed_at,parent_repository,is_fork,github_repository
3,extensao3-2025_2-team_04-Bulicoso-1596a9c51e60,2025-12-05 12:10:31+00:00,2025-12-12 13:21:41+00:00,SilvioGoncalvesXJr/Bulicoso,True,gesad-lab/team_04-Bulicoso
6,extensao3-2025_2-team_08-ExamForge-80e19be1a764,2025-12-05 12:11:42+00:00,2025-12-05 00:55:07+00:00,Gabriel-marques-araujo/ExamForge,True,gesad-lab/team_08-ExamForge
1,extensao3-2025_2-team_02-EstudaAI-1953213cc45b,2025-12-05 12:24:18+00:00,2025-12-05 04:37:37+00:00,HChuvas/EstudaAI,True,gesad-lab/team_02-EstudaAI
7,extensao3-2025_2-team_09-SoFiA-59690ae48d74,2025-12-05 12:55:47+00:00,2025-12-05 12:43:11+00:00,suyanecarvalho/SoFiA,True,gesad-lab/team_09-SoFiA
8,extensao3-2025_2-team_10-Extensao-3-Synapse-1e...,2025-12-05 13:19:58+00:00,2025-12-05 12:21:49+00:00,ianjsm/Extensao-3-Synapse,True,gesad-lab/team_10-Extensao-3-Synapse
5,extensao3-2025_2-team_07-Projeto-Research-Flow...,2025-12-12 11:41:27+00:00,2025-12-11 13:31:01+00:00,GlaucoCiprianoMoreira/Projeto-Research-Flow,True,gesad-lab/team_07-Projeto-Research-Flow
2,extensao3-2025_2-team_03-career-path-ai-671a7f...,2025-12-12 12:12:49+00:00,2025-12-12 11:43:42+00:00,daviolvr/career-path-ai,True,gesad-lab/team_03-career-path-ai
4,extensao3-2025_2-team_05-ecoroteiro-ex3-829bfc...,2025-12-12 13:01:43+00:00,2025-12-12 02:47:21+00:00,jamileehellen/ecoroteiro-ex3,True,gesad-lab/team_05-ecoroteiro-ex3
0,extensao3-2025_2-team_01-software_development_...,2025-12-12 13:03:45+00:00,2025-12-12 03:29:35+00:00,Sergiosampjr/software_development_extension_lll,True,gesad-lab/team_01-software_development_extensi...
9,extensao3-2026_1-team1-foodguard-4f38369906d5,2026-06-22 09:21:26+00:00,2026-06-19 10:39:18+00:00,VictorManoel-Timbo/extensao-3-projeto,True,gesad-lab/team1-foodguard


In [27]:
FORK_WINDOW_DAYS = [1, 3, 7, 14]


def calculate_fork_deadline_windows(
    commits: pd.DataFrame,
    fork_metadata: pd.DataFrame,
    window_days: list[int],
) -> pd.DataFrame:
    rows = []
    for (semester, team_id, repository), team in commits.groupby(
        ["Semestre", "ID_Equipe", "repository"]
    ):
        deadline = fork_metadata.loc[
            fork_metadata["repository"].eq(repository),
            "fork_created_at",
        ].iloc[0]
        for days in window_days:
            window = team[
                team["timestamp"].ge(deadline - pd.Timedelta(days=days))
                & team["timestamp"].lt(deadline)
            ]
            counts = window[CONFIG["author_field"]].dropna().value_counts()
            shares = counts / len(window) if len(window) else pd.Series(dtype=float)
            rows.append(
                {
                    "Semestre": semester,
                    "ID_Equipe": team_id,
                    "repository": repository,
                    "deadline": deadline,
                    "window_days": days,
                    "commit_n": int(len(window)),
                    "author_n": int(len(counts)),
                    "max_author_share": float(shares.max()) if len(shares) else np.nan,
                    "author_gini": gini(counts) if len(counts) else np.nan,
                    "commits_after_deadline": int(team["timestamp"].ge(deadline).sum()),
                }
            )
    return pd.DataFrame(rows)


fork_window_df = calculate_fork_deadline_windows(
    git_commits_df,
    fork_deadline_metadata_df,
    FORK_WINDOW_DAYS,
)
fork_window_summary_df = (
    fork_window_df.groupby("window_days", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        equipes_com_commits=("commit_n", lambda values: values.gt(0).sum()),
        commits=("commit_n", "sum"),
        mediana_autores=("author_n", lambda values: values[values.gt(0)].median()),
        mediana_max_share=("max_author_share", "median"),
        mediana_gini=("author_gini", "median"),
        commits_apos_deadline=("commits_after_deadline", "sum"),
        equipes_com_commits_apos_deadline=(
            "commits_after_deadline",
            lambda values: values.gt(0).sum(),
        ),
    )
)
fork_window_summary_df["cobertura"] = (
    fork_window_summary_df["equipes_com_commits"]
    / fork_window_summary_df["equipes"]
)
fork_window_summary_df.round(3)

,window_days,equipes,equipes_com_commits,commits,mediana_autores,mediana_max_share,mediana_gini,commits_apos_deadline,equipes_com_commits_apos_deadline,cobertura
0,1,14,9,112,3.0,0.583,0.167,11,1,0.643
1,3,14,14,207,2.0,0.562,0.062,11,1,1.000
2,7,14,14,285,3.0,0.489,0.217,11,1,1.000
3,14,14,14,336,3.0,0.513,0.205,11,1,1.000


#### Impacto da nova âncora

Com o fork como deadline:

- 24 horas: 9/14 equipes com commits;
- 72 horas: 14/14;
- 7 e 14 dias: 14/14.

Isso elimina a falsa ausência produzida pelo início agregado de T3 e permite comparar janelas iguais antes do evento de cada equipe.

A auditoria encontrou **um único fork com push posterior à criação**: TEAM_04/2025.2, da equipe Buliçoso.

- semestre: 2025.2;
- apresentação: primeira sessão, em 05/12/2025;
- avaliações registradas: 08:47–08:53 BRT;
- fork criado: 09:10:31 BRT;
- primeiro commit posterior: 09:15:44 BRT, 22min42s após a última avaliação e 5min13s após o fork;
- último commit posterior: 10/12 às 12:50:03 BRT, 5d3h57m01s após a última avaliação;
- `pushed_at` do fork: 12/12 às 10:21:41 BRT, 1d21h31m38s após o último commit;
- commits posteriores no histórico atual: 11;
- committers: Silvio Gonçalves (6) e ItaloVicente (5);
- alteração líquida entre o estado pré-fork e o HEAD final: 12 arquivos, 39 inserções e 25 remoções, além de alterações binárias.

Portanto, **não foi apenas um push tardio**. Segundo `author date`, `committer date` e o diff da árvore, houve modificações reais após a apresentação; posteriormente, essas modificações foram sincronizadas com o fork central.

O merge e o HEAD final foram commitados por ItaloVicente. Entretanto, Git registra autor e committer dos commits, não necessariamente o usuário GitHub que executou o push ou clicou em “Sync fork”. Portanto, **não é possível identificar com segurança quem realizou a sincronização** usando esses dados.

O fork e o repositório pai atualmente apontam para o mesmo HEAD, comportamento compatível com sincronização posterior. Esses 11 commits devem ser excluídos de qualquer janela encerrada na criação do fork. Os outros 13 forks possuem `pushed_at` anterior ao `created_at`, sem sinal de push posterior nos metadados atuais.

In [28]:
def local_commits_after_fork(
    fork_metadata: pd.DataFrame,
    repos_root: Path,
) -> pd.DataFrame:
    rows = []
    for metadata in fork_metadata.to_dict("records"):
        git_dir = repos_root / metadata["repository"] / ".git"
        raw_log = subprocess.check_output(
            ["git", f"--git-dir={git_dir}", "log", "--all", "--format=%H%x09%cI"],
            text=True,
        )
        created_at = metadata["fork_created_at"]
        later_commits = []
        for line in raw_log.splitlines():
            commit_sha, committed_at_text = line.split("\t", 1)
            committed_at = pd.Timestamp(committed_at_text)
            if committed_at > created_at:
                later_commits.append((commit_sha, committed_at))
        later_commits.sort(key=lambda item: item[1])
        pushed_at = metadata["current_pushed_at"]
        rows.append(
            {
                "repository": metadata["repository"],
                "fork_created_at": created_at,
                "current_pushed_at": pushed_at,
                "push_after_fork_creation": pushed_at > created_at,
                "commits_timestamped_after_creation": len(later_commits),
                "first_commit_after": later_commits[0][1] if later_commits else pd.NaT,
                "first_commit_delay": later_commits[0][1] - created_at if later_commits else pd.NaT,
                "last_commit_after": later_commits[-1][1] if later_commits else pd.NaT,
                "last_commit_delay": later_commits[-1][1] - created_at if later_commits else pd.NaT,
                "push_delay": pushed_at - created_at,
            }
        )
    return pd.DataFrame(rows)


post_fork_audit_df = local_commits_after_fork(
    fork_deadline_metadata_df,
    PROJECT_ROOT / "data/raw/repos_cache",
)
confirmed_post_fork_updates_df = post_fork_audit_df[
    post_fork_audit_df["push_after_fork_creation"]
]
assert len(confirmed_post_fork_updates_df) == 1
confirmed_post_fork_updates_df

,repository,fork_created_at,current_pushed_at,push_after_fork_creation,commits_timestamped_after_creation,first_commit_after,first_commit_delay,last_commit_after,last_commit_delay,push_delay
3,extensao3-2025_2-team_04-Bulicoso-1596a9c51e60,2025-12-05 12:10:31+00:00,2025-12-12 13:21:41+00:00,True,11,2025-12-05 09:15:44-03:00,0 days 00:05:13,2025-12-10 12:50:03-03:00,5 days 03:39:32,7 days 01:11:10


In [29]:
team04_repository = "extensao3-2025_2-team_04-Bulicoso-1596a9c51e60"
team04_git_dir = PROJECT_ROOT / "data/raw/repos_cache" / team04_repository / ".git"
team04_created_at = fork_deadline_metadata_df.loc[
    fork_deadline_metadata_df["repository"].eq(team04_repository),
    "fork_created_at",
].iloc[0]

team04_log = subprocess.check_output(
    [
        "git",
        f"--git-dir={team04_git_dir}",
        "log",
        "--all",
        "--format=%H%x09%cI%x09%cN",
    ],
    text=True,
)
team04_later_commits = []
for line in team04_log.splitlines():
    commit_sha, committed_at_text, committer_name = line.split("\t", 2)
    committed_at = pd.Timestamp(committed_at_text)
    if committed_at > team04_created_at:
        team04_later_commits.append(
            {
                "commit_sha": commit_sha,
                "committed_at": committed_at,
                "committer_name": committer_name,
            }
        )
team04_later_commits_df = pd.DataFrame(team04_later_commits)

forms_2025_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/forms/2025.2/avaliadores.csv"
)
team04_evaluations_df = forms_2025_df[
    forms_2025_df["To which group do these scores refer?"].str.contains(
        "Buliçoso",
        case=False,
        na=False,
    )
].copy()
team04_evaluations_df["evaluation_at"] = pd.to_datetime(
    team04_evaluations_df["Timestamp"],
    format="mixed",
)
team04_t3_evaluations_df = team04_evaluations_df[
    team04_evaluations_df["evaluation_at"].dt.date.eq(pd.Timestamp("2025-12-05").date())
]

team04_audit = {
    "semester": "2025.2",
    "presentation_session": "primeira sessão (05/12/2025)",
    "evaluation_first": team04_t3_evaluations_df["evaluation_at"].min(),
    "evaluation_last": team04_t3_evaluations_df["evaluation_at"].max(),
    "fork_created_at_utc": team04_created_at,
    "post_fork_committers": team04_later_commits_df["committer_name"].value_counts().to_dict(),
    "post_fork_commits": len(team04_later_commits_df),
    "synchronization_actor_identifiable": False,
}
team04_audit

{'semester': '2025.2',
 'presentation_session': 'primeira sessão (05/12/2025)',
 'evaluation_first': Timestamp('2025-12-05 08:47:46'),
 'evaluation_last': Timestamp('2025-12-05 08:53:02'),
 'fork_created_at_utc': Timestamp('2025-12-05 12:10:31+0000', tz='UTC'),
 'post_fork_committers': {'Silvio Gonçalves': 6, 'ItaloVicente': 5},
 'post_fork_commits': 11,
 'synchronization_actor_identifiable': False}

In [30]:
presentation_last = pd.Timestamp(
    team04_t3_evaluations_df["evaluation_at"].max(),
    tz="America/Fortaleza",
)
fork_local = team04_created_at.tz_convert("America/Fortaleza")
first_commit_local = team04_later_commits_df["committed_at"].min()
last_commit_local = team04_later_commits_df["committed_at"].max()
push_local = fork_deadline_metadata_df.loc[
    fork_deadline_metadata_df["repository"].eq(team04_repository),
    "current_pushed_at",
].iloc[0].tz_convert("America/Fortaleza")

pre_fork_commit = subprocess.check_output(
    [
        "git",
        f"--git-dir={team04_git_dir}",
        "rev-parse",
        f"{team04_later_commits_df.sort_values('committed_at').iloc[0]['commit_sha']}^",
    ],
    text=True,
).strip()
final_head = subprocess.check_output(
    ["git", f"--git-dir={team04_git_dir}", "rev-parse", "HEAD"],
    text=True,
).strip()
tree_change_summary = subprocess.check_output(
    [
        "git",
        f"--git-dir={team04_git_dir}",
        "diff",
        "--shortstat",
        f"{pre_fork_commit}..{final_head}",
    ],
    text=True,
).strip()

team04_chronology = {
    "presentation_last": presentation_last,
    "fork_created": fork_local,
    "first_post_presentation_commit": first_commit_local,
    "last_post_presentation_commit": last_commit_local,
    "fork_pushed_at": push_local,
    "presentation_to_first_commit": first_commit_local - presentation_last,
    "presentation_to_last_commit": last_commit_local - presentation_last,
    "last_commit_to_fork_push": push_local - last_commit_local,
    "tree_change_summary": tree_change_summary,
    "conclusion": "modificações reais após a apresentação, seguidas por sincronização posterior",
}
team04_chronology

{'presentation_last': Timestamp('2025-12-05 08:53:02-0300', tz='America/Fortaleza'),
 'fork_created': Timestamp('2025-12-05 09:10:31-0300', tz='America/Fortaleza'),
 'first_post_presentation_commit': Timestamp('2025-12-05 09:15:44-0300', tz='UTC-03:00'),
 'last_post_presentation_commit': Timestamp('2025-12-10 12:50:03-0300', tz='UTC-03:00'),
 'fork_pushed_at': Timestamp('2025-12-12 10:21:41-0300', tz='America/Fortaleza'),
 'presentation_to_first_commit': Timedelta('0 days 00:22:42'),
 'presentation_to_last_commit': Timedelta('5 days 03:57:01'),
 'last_commit_to_fork_push': Timedelta('1 days 21:31:38'),
 'tree_change_summary': '12 files changed, 39 insertions(+), 25 deletions(-)',
 'conclusion': 'modificações reais após a apresentação, seguidas por sincronização posterior'}

## 4. M3a — Participação dos commits na fase final

A âncora $P_i$ é o **último voto T3 registrado para a equipe**. Ela representa o fim operacional observado da sessão.

As fases são:

$$
W_i^{-}=[P_i-7d,P_i)
$$

$$
W_i^{+}=[P_i,P_i+7d)
$$

O indicador principal é:

$$
\mathrm{FinalCommitShare}_i=
\frac{C_i(W_i^{-})+C_i(W_i^{+})}
{C_i(t<P_i+7d)}
$$

Também são reportadas separadamente as parcelas pré e pós. O denominador termina em $P_i+7d$ para não usar informação futura.

A fonte é a branch padrão do repositório original em mirror read-only. A inclusão temporal usa `committer date`. TEAM_04/2025.2 é a única extensão pós-apresentação declarada como autorizada.

In [32]:
def evaluator_vote_anchors(
    project_root: Path,
    semesters: list[str],
) -> pd.DataFrame:
    frames = []
    for semester in semesters:
        frame = pd.read_csv(
            project_root / f"data/processed/forms/{semester}/avaliadores.csv"
        )
        frame["ID_Equipe"] = (
            frame["To which group do these scores refer?"]
            .str.extract(r"Group\s+(\d+)", expand=False)
            .astype(int)
            .map(lambda value: f"TEAM_{value:02d}")
        )
        frame["vote_at"] = pd.to_datetime(
            frame["Timestamp"],
            format="mixed",
        ).dt.tz_localize("America/Fortaleza")
        start, end = EVALUATOR_TEMPORAL_CUTS[semester]["T3"]
        is_t3 = frame["vote_at"].dt.date.between(
            pd.Timestamp(start).date(),
            pd.Timestamp(end).date(),
        )
        anchors = (
            frame[is_t3]
            .groupby("ID_Equipe", as_index=False)
            .agg(
                first_vote_at=("vote_at", "min"),
                presentation_anchor=("vote_at", "max"),
                evaluator_vote_n=("vote_at", "size"),
            )
            .assign(Semestre=semester)
        )
        frames.append(anchors)
    return pd.concat(frames, ignore_index=True)


presentation_anchors_df = evaluator_vote_anchors(
    PROJECT_ROOT,
    ["2025.2", "2026.1"],
)
repository_keys_df = git_commits_df[
    ["Semestre", "ID_Equipe", "repository"]
].drop_duplicates()
presentation_anchors_df = presentation_anchors_df.merge(
    repository_keys_df,
    on=TEAM_KEY,
    validate="one_to_one",
)

anchor_audit_df = pd.DataFrame(
    [
        {
            "verificação": "14 equipes possuem âncora T3",
            "resultado": len(presentation_anchors_df) == 14,
        },
        {
            "verificação": "Uma âncora por equipe-semestre",
            "resultado": not presentation_anchors_df.duplicated(TEAM_KEY).any(),
        },
        {
            "verificação": "Todas as âncoras têm votos",
            "resultado": presentation_anchors_df["evaluator_vote_n"].gt(0).all(),
        },
        {
            "verificação": "Somente TEAM_04/2025.2 possui extensão autorizada declarada",
            "resultado": CONFIG["authorized_post_presentation_extensions"]
            == ["2025.2|TEAM_04"],
        },
    ]
)

display(anchor_audit_df)
presentation_anchors_df.sort_values(
    ["Semestre", "presentation_anchor", "ID_Equipe"]
)

,verificação,resultado
0,14 equipes possuem âncora T3,True
1,Uma âncora por equipe-semestre,True
2,Todas as âncoras têm votos,True
3,Somente TEAM_04/2025.2 possui extensão autoriz...,True


,ID_Equipe,first_vote_at,presentation_anchor,evaluator_vote_n,Semestre,repository
3,TEAM_04,2025-12-05 08:47:46-03:00,2025-12-05 08:53:02-03:00,3,2025.2,extensao3-2025_2-team_04-Bulicoso-1596a9c51e60
6,TEAM_08,2025-12-05 09:14:30-03:00,2025-12-05 09:20:15-03:00,3,2025.2,extensao3-2025_2-team_08-ExamForge-80e19be1a764
1,TEAM_02,2025-12-05 09:47:52-03:00,2025-12-05 09:54:04-03:00,3,2025.2,extensao3-2025_2-team_02-EstudaAI-1953213cc45b
7,TEAM_09,2025-12-05 10:17:02-03:00,2025-12-05 10:21:51-03:00,3,2025.2,extensao3-2025_2-team_09-SoFiA-59690ae48d74
8,TEAM_10,2025-12-05 10:35:06-03:00,2025-12-05 10:38:16-03:00,2,2025.2,extensao3-2025_2-team_10-Extensao-3-Synapse-1e...
2,TEAM_03,2025-12-12 09:28:59-03:00,2025-12-12 09:43:55-03:00,4,2025.2,extensao3-2025_2-team_03-career-path-ai-671a7f...
5,TEAM_07,2025-12-12 08:56:03-03:00,2025-12-12 09:44:15-03:00,4,2025.2,extensao3-2025_2-team_07-Projeto-Research-Flow...
0,TEAM_01,2025-12-12 09:53:40-03:00,2025-12-12 10:29:05-03:00,4,2025.2,extensao3-2025_2-team_01-software_development_...
4,TEAM_05,2025-12-12 10:12:01-03:00,2025-12-12 10:30:13-03:00,4,2025.2,extensao3-2025_2-team_05-ecoroteiro-ex3-829bfc...
11,TEAM_03,2026-06-19 08:55:33-03:00,2026-06-19 09:07:49-03:00,3,2026.1,extensao3-2026_1-team3-interview-17c184cfaa5e


In [34]:
PARENT_CACHE_DIR = PROJECT_ROOT / "data/raw/repos_parent_cache"


def commit_churn(git_dir: Path, commit_sha: str) -> tuple[int, int]:
    output = subprocess.check_output(
        [
            "git",
            f"--git-dir={git_dir}",
            "diff-tree",
            "--root",
            "--no-commit-id",
            "--numstat",
            "-r",
            "--find-renames",
            commit_sha,
        ],
        text=True,
    )
    churn = 0
    binary_events = 0
    for line in output.splitlines():
        fields = line.split("\t")
        if len(fields) < 3:
            continue
        added, deleted = fields[:2]
        if added == "-" or deleted == "-":
            binary_events += 1
        else:
            churn += int(added) + int(deleted)
    return churn, binary_events


def parent_default_branch_commits(
    git_dir: Path,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    default_branch = subprocess.check_output(
        ["git", f"--git-dir={git_dir}", "symbolic-ref", "--short", "HEAD"],
        text=True,
    ).strip()
    output = subprocess.check_output(
        [
            "git",
            f"--git-dir={git_dir}",
            "log",
            default_branch,
            "--format=%H%x09%cI%x09%aE%x09%aN",
        ],
        text=True,
    )
    rows = []
    for line in output.splitlines():
        commit_sha, committed_at_text, author_email, author_name = line.split("\t", 3)
        committed_at = pd.Timestamp(committed_at_text)
        if not start <= committed_at < end:
            continue
        churn, binary_events = commit_churn(git_dir, commit_sha)
        author_identity = author_email.strip().lower() or author_name.strip().lower()
        rows.append(
            {
                "commit_sha": commit_sha,
                "committed_at": committed_at,
                "author_key": sha256(author_identity.encode("utf-8")).hexdigest()[:12],
                "churn": churn,
                "binary_events": binary_events,
                "default_branch": default_branch,
            }
        )
    result = pd.DataFrame(rows)
    if not result.empty:
        result["committed_at"] = pd.to_datetime(result["committed_at"], utc=True)
    return result


def calculate_final_phase_metrics(
    anchors: pd.DataFrame,
    parent_cache_dir: Path,
    days_before: int,
    days_after: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    commit_frames = []
    authorized = set(CONFIG["authorized_post_presentation_extensions"])
    for anchor in anchors.to_dict("records"):
        presentation = anchor["presentation_anchor"]
        analysis_start = presentation - pd.Timedelta(days=days_before)
        analysis_end = presentation + pd.Timedelta(days=days_after)
        git_dir = parent_cache_dir / f"{anchor['repository']}.git"
        commits = parent_default_branch_commits(git_dir, analysis_start, analysis_end)
        if not commits.empty:
            presentation_utc = presentation.tz_convert("UTC")
            commits = commits.assign(
                Semestre=anchor["Semestre"],
                ID_Equipe=anchor["ID_Equipe"],
                repository=anchor["repository"],
                presentation_anchor=presentation,
                phase=np.where(
                    commits["committed_at"].lt(presentation_utc),
                    "pre",
                    "post",
                ),
            )
            commit_frames.append(commits)
        for phase, phase_start, phase_end in (
            ("pre", analysis_start, presentation),
            ("post", presentation, analysis_end),
        ):
            current = commits[commits["phase"].eq(phase)] if not commits.empty else commits
            counts = (
                current["author_key"].value_counts()
                if not current.empty
                else pd.Series(dtype=int)
            )
            shares = counts / len(current) if len(current) else pd.Series(dtype=float)
            team_key = f"{anchor['Semestre']}|{anchor['ID_Equipe']}"
            active_days = (
                current["committed_at"]
                .dt.tz_convert("America/Fortaleza")
                .dt.date.nunique()
                if len(current)
                else 0
            )
            metric_rows.append(
                {
                    "Semestre": anchor["Semestre"],
                    "ID_Equipe": anchor["ID_Equipe"],
                    "repository": anchor["repository"],
                    "presentation_anchor": presentation,
                    "phase": phase,
                    "window_start": phase_start,
                    "window_end": phase_end,
                    "exposure_days": (phase_end - phase_start).total_seconds() / 86400,
                    "commit_n": int(len(current)),
                    "commits_per_day": len(current) / 7,
                    "active_days": int(active_days),
                    "author_n": int(len(counts)),
                    "max_author_share": float(shares.max()) if len(shares) else np.nan,
                    "author_gini": gini(counts) if len(counts) else np.nan,
                    "churn": int(current["churn"].sum()) if len(current) else 0,
                    "binary_events": int(current["binary_events"].sum()) if len(current) else 0,
                    "authorized_extension": phase == "post" and team_key in authorized,
                }
            )
    metrics = pd.DataFrame(metric_rows)
    commits = pd.concat(commit_frames, ignore_index=True) if commit_frames else pd.DataFrame()
    return metrics, commits


final_phase_metrics_df, final_phase_commits_private_df = calculate_final_phase_metrics(
    presentation_anchors_df,
    PARENT_CACHE_DIR,
    CONFIG["window_days_before"],
    CONFIG["window_days_after"],
)

final_phase_metrics_df.sort_values(
    ["Semestre", "presentation_anchor", "ID_Equipe", "phase"]
).round(3)

,Semestre,ID_Equipe,repository,presentation_anchor,phase,window_start,window_end,exposure_days,commit_n,commits_per_day,active_days,author_n,max_author_share,author_gini,churn,binary_events,authorized_extension
7,2025.2,TEAM_04,extensao3-2025_2-team_04-Bulicoso-1596a9c51e60,2025-12-05 08:53:02-03:00,post,2025-12-05 08:53:02-03:00,2025-12-12 08:53:02-03:00,7.0,12,1.714,3,3,0.500,0.278,92,7,True
6,2025.2,TEAM_04,extensao3-2025_2-team_04-Bulicoso-1596a9c51e60,2025-12-05 08:53:02-03:00,pre,2025-11-28 08:53:02-03:00,2025-12-05 08:53:02-03:00,7.0,27,3.857,4,5,0.704,0.563,11941,41,False
13,2025.2,TEAM_08,extensao3-2025_2-team_08-ExamForge-80e19be1a764,2025-12-05 09:20:15-03:00,post,2025-12-05 09:20:15-03:00,2025-12-12 09:20:15-03:00,7.0,6,0.857,1,2,0.833,0.333,247,2,False
12,2025.2,TEAM_08,extensao3-2025_2-team_08-ExamForge-80e19be1a764,2025-12-05 09:20:15-03:00,pre,2025-11-28 09:20:15-03:00,2025-12-05 09:20:15-03:00,7.0,8,1.143,5,3,0.500,0.250,1751,4,False
3,2025.2,TEAM_02,extensao3-2025_2-team_02-EstudaAI-1953213cc45b,2025-12-05 09:54:04-03:00,post,2025-12-05 09:54:04-03:00,2025-12-12 09:54:04-03:00,7.0,0,0.000,0,0,NaN,NaN,0,0,False
2,2025.2,TEAM_02,extensao3-2025_2-team_02-EstudaAI-1953213cc45b,2025-12-05 09:54:04-03:00,pre,2025-11-28 09:54:04-03:00,2025-12-05 09:54:04-03:00,7.0,109,15.571,7,5,0.477,0.418,49261,45,False
15,2025.2,TEAM_09,extensao3-2025_2-team_09-SoFiA-59690ae48d74,2025-12-05 10:21:51-03:00,post,2025-12-05 10:21:51-03:00,2025-12-12 10:21:51-03:00,7.0,0,0.000,0,0,NaN,NaN,0,0,False
14,2025.2,TEAM_09,extensao3-2025_2-team_09-SoFiA-59690ae48d74,2025-12-05 10:21:51-03:00,pre,2025-11-28 10:21:51-03:00,2025-12-05 10:21:51-03:00,7.0,52,7.429,4,5,0.442,0.415,28451,8,False
17,2025.2,TEAM_10,extensao3-2025_2-team_10-Extensao-3-Synapse-1e...,2025-12-05 10:38:16-03:00,post,2025-12-05 10:38:16-03:00,2025-12-12 10:38:16-03:00,7.0,0,0.000,0,0,NaN,NaN,0,0,False
16,2025.2,TEAM_10,extensao3-2025_2-team_10-Extensao-3-Synapse-1e...,2025-12-05 10:38:16-03:00,pre,2025-11-28 10:38:16-03:00,2025-12-05 10:38:16-03:00,7.0,7,1.000,3,3,0.429,0.190,2202,5,False


In [35]:
final_phase_summary_df = (
    final_phase_metrics_df.groupby(["Semestre", "phase"], as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        equipes_com_commits=("commit_n", lambda values: values.gt(0).sum()),
        commits=("commit_n", "sum"),
        mediana_commits=("commit_n", "median"),
        mediana_commits_por_dia=("commits_per_day", "median"),
        mediana_dias_ativos=("active_days", "median"),
        mediana_autores=("author_n", lambda values: values[values.gt(0)].median()),
        mediana_maior_participacao=("max_author_share", "median"),
        mediana_gini=("author_gini", "median"),
        churn=("churn", "sum"),
    )
)
final_phase_summary_df["cobertura"] = (
    final_phase_summary_df["equipes_com_commits"]
    / final_phase_summary_df["equipes"]
)

post_activity_df = final_phase_metrics_df[
    final_phase_metrics_df["phase"].eq("post")
].copy()
post_activity_audit_df = pd.DataFrame(
    [
        {
            "verificação": "Todas as 14 equipes têm atividade pré-apresentação",
            "resultado": final_phase_metrics_df.loc[
                final_phase_metrics_df["phase"].eq("pre"),
                "commit_n",
            ].gt(0).all(),
        },
        {
            "verificação": "Sete equipes têm commits pós-apresentação",
            "resultado": post_activity_df["commit_n"].gt(0).sum() == 7,
        },
        {
            "verificação": "Extensão autorizada declarada somente para TEAM_04/2025.2",
            "resultado": post_activity_df.loc[
                post_activity_df["authorized_extension"],
                TEAM_KEY,
            ].to_dict("records")
            == [{"Semestre": "2025.2", "ID_Equipe": "TEAM_04"}],
        },
    ]
)

display(post_activity_audit_df)
display(final_phase_summary_df.round(3))
post_activity_df.loc[
    post_activity_df["commit_n"].gt(0),
    TEAM_KEY
    + [
        "commit_n",
        "active_days",
        "author_n",
        "max_author_share",
        "churn",
        "authorized_extension",
    ],
].sort_values(TEAM_KEY).round(3)

,verificação,resultado
0,Todas as 14 equipes têm atividade pré-apresent...,True
1,Sete equipes têm commits pós-apresentação,True
2,Extensão autorizada declarada somente para TEA...,True


,Semestre,phase,equipes,equipes_com_commits,commits,mediana_commits,mediana_commits_por_dia,mediana_dias_ativos,mediana_autores,mediana_maior_participacao,mediana_gini,churn,cobertura
0,2025.2,post,9,4,21,0.0,0.000,0.0,1.5,0.917,0.139,766,0.444
1,2025.2,pre,9,9,230,9.0,1.286,3.0,3.0,0.500,0.222,1492660,1.000
2,2026.1,post,5,3,7,1.0,0.143,1.0,1.0,1.000,0.000,43924,0.600
3,2026.1,pre,5,5,64,14.0,2.000,4.0,4.0,0.500,0.214,58348,1.000


,Semestre,ID_Equipe,commit_n,active_days,author_n,max_author_share,churn,authorized_extension
7,2025.2,TEAM_04,12,3,3,0.500,92,True
9,2025.2,TEAM_05,2,1,1,1.000,56,False
11,2025.2,TEAM_07,1,1,1,1.000,371,False
13,2025.2,TEAM_08,6,1,2,0.833,247,False
23,2026.1,TEAM_03,1,1,1,1.000,1,False
25,2026.1,TEAM_04,3,1,1,1.000,43877,False
27,2026.1,TEAM_05,3,1,2,0.667,46,False


In [40]:
def default_branch_commit_times(git_dir: Path) -> tuple[str, pd.Series]:
    default_branch = subprocess.check_output(
        ["git", f"--git-dir={git_dir}", "symbolic-ref", "--short", "HEAD"],
        text=True,
    ).strip()
    output = subprocess.check_output(
        [
            "git",
            f"--git-dir={git_dir}",
            "log",
            default_branch,
            "--format=%cI",
        ],
        text=True,
    )
    timestamps = pd.Series(
        pd.to_datetime(output.splitlines(), utc=True),
        dtype="datetime64[ns, UTC]",
    )
    return default_branch, timestamps


phase_counts_df = (
    final_phase_metrics_df.pivot(
        index=TEAM_KEY,
        columns="phase",
        values="commit_n",
    )
    .reset_index()
    .rename(columns={"pre": "pre_commit_n", "post": "post_commit_n"})
)

share_rows = []
for anchor in presentation_anchors_df.to_dict("records"):
    git_dir = PARENT_CACHE_DIR / f"{anchor['repository']}.git"
    default_branch, timestamps = default_branch_commit_times(git_dir)
    denominator_end = anchor["presentation_anchor"] + pd.Timedelta(
        days=CONFIG["window_days_after"]
    )
    observed = timestamps[timestamps.lt(denominator_end.tz_convert("UTC"))]
    share_rows.append(
        {
            "Semestre": anchor["Semestre"],
            "ID_Equipe": anchor["ID_Equipe"],
            "repository": anchor["repository"],
            "presentation_anchor": anchor["presentation_anchor"],
            "denominator_end": denominator_end,
            "default_branch": default_branch,
            "total_commit_n": int(len(observed)),
            "history_first_commit": observed.min() if len(observed) else pd.NaT,
        }
    )

final_commit_share_df = pd.DataFrame(share_rows).merge(
    phase_counts_df,
    on=TEAM_KEY,
    validate="one_to_one",
)
final_commit_share_df["final_window_commit_n"] = (
    final_commit_share_df["pre_commit_n"]
    + final_commit_share_df["post_commit_n"]
)
for numerator in ("pre_commit_n", "post_commit_n", "final_window_commit_n"):
    output_column = numerator.replace("commit_n", "share")
    final_commit_share_df[output_column] = (
        final_commit_share_df[numerator]
        / final_commit_share_df["total_commit_n"]
    )
final_commit_share_df["small_denominator_lt_10"] = final_commit_share_df[
    "total_commit_n"
].lt(10)

final_commit_share_df[
    TEAM_KEY
    + [
        "total_commit_n",
        "pre_commit_n",
        "post_commit_n",
        "final_window_commit_n",
        "pre_share",
        "post_share",
        "final_window_share",
        "small_denominator_lt_10",
    ]
].sort_values(TEAM_KEY).round(3)

,Semestre,ID_Equipe,total_commit_n,pre_commit_n,post_commit_n,final_window_commit_n,pre_share,post_share,final_window_share,small_denominator_lt_10
0,2025.2,TEAM_01,19,11,0,11,0.579,0.000,0.579,False
1,2025.2,TEAM_02,201,109,0,109,0.542,0.000,0.542,False
2,2025.2,TEAM_03,9,9,0,9,1.000,0.000,1.000,True
3,2025.2,TEAM_04,63,27,12,39,0.429,0.190,0.619,False
4,2025.2,TEAM_05,3,1,2,3,0.333,0.667,1.000,True
5,2025.2,TEAM_07,74,6,1,7,0.081,0.014,0.095,False
6,2025.2,TEAM_08,60,8,6,14,0.133,0.100,0.233,False
7,2025.2,TEAM_09,107,52,0,52,0.486,0.000,0.486,False
8,2025.2,TEAM_10,18,7,0,7,0.389,0.000,0.389,False
9,2026.1,TEAM_01,3,1,0,1,0.333,0.000,0.333,True


In [41]:
share_summary_by_semester_df = (
    final_commit_share_df.groupby("Semestre", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        mediana_participacao_final=("final_window_share", "median"),
        q1_participacao_final=("final_window_share", lambda values: values.quantile(0.25)),
        q3_participacao_final=("final_window_share", lambda values: values.quantile(0.75)),
        minimo_participacao_final=("final_window_share", "min"),
        maximo_participacao_final=("final_window_share", "max"),
        commits_totais=("total_commit_n", "sum"),
        commits_fase_final=("final_window_commit_n", "sum"),
        equipes_denominador_menor_10=("small_denominator_lt_10", "sum"),
    )
)
share_summary_by_semester_df["participacao_ponderada_final"] = (
    share_summary_by_semester_df["commits_fase_final"]
    / share_summary_by_semester_df["commits_totais"]
)

share_summary_pooled_df = pd.DataFrame(
    [
        {
            "Semestre": "Todos",
            "equipes": len(final_commit_share_df),
            "mediana_participacao_final": final_commit_share_df["final_window_share"].median(),
            "q1_participacao_final": final_commit_share_df["final_window_share"].quantile(0.25),
            "q3_participacao_final": final_commit_share_df["final_window_share"].quantile(0.75),
            "minimo_participacao_final": final_commit_share_df["final_window_share"].min(),
            "maximo_participacao_final": final_commit_share_df["final_window_share"].max(),
            "commits_totais": int(final_commit_share_df["total_commit_n"].sum()),
            "commits_fase_final": int(final_commit_share_df["final_window_commit_n"].sum()),
            "equipes_denominador_menor_10": int(final_commit_share_df["small_denominator_lt_10"].sum()),
            "participacao_ponderada_final": (
                final_commit_share_df["final_window_commit_n"].sum()
                / final_commit_share_df["total_commit_n"].sum()
            ),
        }
    ]
)
final_commit_share_summary_df = pd.concat(
    [share_summary_by_semester_df, share_summary_pooled_df],
    ignore_index=True,
)

share_checks = {
    "fourteen_team_shares": len(final_commit_share_df) == 14,
    "all_denominators_positive": final_commit_share_df["total_commit_n"].gt(0).all(),
    "final_counts_do_not_exceed_total": final_commit_share_df["final_window_commit_n"].le(final_commit_share_df["total_commit_n"]).all(),
    "all_shares_between_zero_and_one": final_commit_share_df[["pre_share", "post_share", "final_window_share"]].apply(lambda column: column.between(0, 1).all()).all(),
    "shares_add_exactly": np.allclose(
        final_commit_share_df["pre_share"] + final_commit_share_df["post_share"],
        final_commit_share_df["final_window_share"],
    ),
}
assert all(share_checks.values()), share_checks

display(pd.Series(share_checks, name="resultado"))
final_commit_share_summary_df.round(3)

fourteen_team_shares                True
all_denominators_positive           True
final_counts_do_not_exceed_total    True
all_shares_between_zero_and_one     True
shares_add_exactly                  True
Name: resultado, dtype: bool

,Semestre,equipes,mediana_participacao_final,q1_participacao_final,q3_participacao_final,minimo_participacao_final,maximo_participacao_final,commits_totais,commits_fase_final,equipes_denominador_menor_10,participacao_ponderada_final
0,2025.2,9,0.542,0.389,0.619,0.095,1.00,554,251,2,0.453
1,2026.1,5,0.286,0.225,0.333,0.137,0.63,295,71,1,0.241
2,Todos,14,0.437,0.246,0.609,0.095,1.00,849,322,3,0.379


### Interpretação da participação final

A proporção é mais comparável que a contagem bruta, mas não deve ser lida sem o denominador:

- uma equipe com poucos commits totais pode apresentar participação final igual a 100%;
- participação alta indica concentração temporal, não volume alto, rework ou mau planejamento;
- `pre_share` e `post_share` devem permanecer visíveis;
- resultados devem ser estratificados por semestre;
- a mediana entre equipes e a proporção ponderada por commits respondem perguntas diferentes.

O resultado principal é a distribuição de `final_window_share`. Contagem total, dias ativos e concentração de autoria explicam o contexto de cada proporção.

In [42]:
share_figure_df = final_commit_share_df.sort_values(TEAM_KEY).copy()
share_figure_df["team_semester"] = (
    share_figure_df["Semestre"] + "/" + share_figure_df["ID_Equipe"]
)
share_figure = go.Figure()
for column, label in (
    ("pre_share", "7 dias antes"),
    ("post_share", "7 dias depois"),
):
    share_figure.add_trace(
        go.Bar(
            x=share_figure_df["team_semester"],
            y=share_figure_df[column],
            name=label,
            customdata=share_figure_df[
                ["total_commit_n", "final_window_commit_n"]
            ],
            hovertemplate=(
                "%{x}<br>Participação: %{y:.1%}"
                "<br>Commits totais: %{customdata[0]}"
                "<br>Commits na fase final: %{customdata[1]}"
                "<extra></extra>"
            ),
        )
    )
share_figure.update_layout(
    title="Participação dos commits na fase final",
    xaxis_title="Equipe-semestre",
    yaxis_title="Proporção dos commits acumulados até P+7d",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1],
    barmode="stack",
)
display(HTML(share_figure.to_html(include_plotlyjs="cdn", full_html=False)))

## 5. M3b — Concentração de autoria na fase final

M3b usa as mesmas fases `pre` e `post` de M3a, mas responde quem concentrou a atividade. Para cada fase, reporta:

- commits e dias ativos;
- autores ativos;
- maior participação de um autor;
- Gini complementar;
- churn apenas como contexto.

M3b não deve ser combinado em um escore único com M3a. Temporalidade e concentração são dimensões distintas.

## 6. M3c — Dinâmica temporal da atividade e autoria

### 6.1 Janelas móveis sobrepostas

Para observar **quando** atividade e concentração aumentam, usamos janelas móveis de sete dias com passo diário. Cada ponto $d$ representa:

$$
[P_i+d-7d,\ P_i+d)
$$

com $d\in[-21,+7]$.

Janelas consecutivas compartilham seis dos sete dias. Os pontos são autocorrelacionados e servem para visualização descritiva, não para testes que presumem independência.

In [49]:
ROLLING_WINDOW_DAYS = 7
ROLLING_END_OFFSETS = range(-21, 8)


def parent_commit_identities(
    git_dir: Path,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    default_branch = subprocess.check_output(
        ["git", f"--git-dir={git_dir}", "symbolic-ref", "--short", "HEAD"],
        text=True,
    ).strip()
    output = subprocess.check_output(
        [
            "git",
            f"--git-dir={git_dir}",
            "log",
            default_branch,
            "--format=%cI%x09%aE%x09%aN",
        ],
        text=True,
    )
    rows = []
    for line in output.splitlines():
        committed_at_text, author_email, author_name = line.split("\t", 2)
        committed_at = pd.Timestamp(committed_at_text)
        if not start <= committed_at < end:
            continue
        author_identity = author_email.strip().lower() or author_name.strip().lower()
        rows.append(
            {
                "committed_at": committed_at,
                "author_key": sha256(author_identity.encode("utf-8")).hexdigest()[:12],
            }
        )
    result = pd.DataFrame(rows)
    if not result.empty:
        result["committed_at"] = pd.to_datetime(result["committed_at"], utc=True)
    return result


def rolling_concentration_metrics(
    anchors: pd.DataFrame,
    parent_cache_dir: Path,
    window_days: int,
    end_offsets: range,
) -> pd.DataFrame:
    rows = []
    for anchor in anchors.to_dict("records"):
        presentation = anchor["presentation_anchor"]
        earliest_start = presentation + pd.Timedelta(
            days=min(end_offsets) - window_days
        )
        latest_end = presentation + pd.Timedelta(days=max(end_offsets))
        git_dir = parent_cache_dir / f"{anchor['repository']}.git"
        commits = parent_commit_identities(
            git_dir,
            earliest_start,
            latest_end,
        )
        for end_offset in end_offsets:
            window_end = presentation + pd.Timedelta(days=end_offset)
            window_start = window_end - pd.Timedelta(days=window_days)
            if commits.empty:
                current = commits
            else:
                current = commits[
                    commits["committed_at"].ge(window_start.tz_convert("UTC"))
                    & commits["committed_at"].lt(window_end.tz_convert("UTC"))
                ]
            counts = (
                current["author_key"].value_counts()
                if not current.empty
                else pd.Series(dtype=int)
            )
            shares = counts / len(current) if len(current) else pd.Series(dtype=float)
            rows.append(
                {
                    "Semestre": anchor["Semestre"],
                    "ID_Equipe": anchor["ID_Equipe"],
                    "window_end_day": end_offset,
                    "window_start": window_start,
                    "window_end": window_end,
                    "commit_n": int(len(current)),
                    "active_days": int(
                        current["committed_at"]
                        .dt.tz_convert("America/Fortaleza")
                        .dt.date.nunique()
                    )
                    if len(current)
                    else 0,
                    "author_n": int(len(counts)),
                    "max_author_share": float(shares.max()) if len(shares) else np.nan,
                    "author_gini": gini(counts) if len(counts) else np.nan,
                }
            )
    return pd.DataFrame(rows)


rolling_metrics_df = rolling_concentration_metrics(
    presentation_anchors_df,
    PARENT_CACHE_DIR,
    ROLLING_WINDOW_DAYS,
    ROLLING_END_OFFSETS,
)
rolling_summary_df = (
    rolling_metrics_df.groupby(["Semestre", "window_end_day"], as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        equipes_ativas=("commit_n", lambda values: values.gt(0).sum()),
        commits=("commit_n", "sum"),
        mediana_commits=("commit_n", "median"),
        mediana_dias_ativos=("active_days", "median"),
        mediana_maior_participacao=("max_author_share", "median"),
        mediana_gini=("author_gini", "median"),
    )
)
rolling_summary_df["cobertura"] = (
    rolling_summary_df["equipes_ativas"] / rolling_summary_df["equipes"]
)

rolling_summary_df[
    rolling_summary_df["window_end_day"].isin([-14, -7, 0, 1, 3, 7])
].round(3)

,Semestre,window_end_day,equipes,equipes_ativas,commits,mediana_commits,mediana_dias_ativos,mediana_maior_participacao,mediana_gini,cobertura
7,2025.2,-14,9,4,51,0.0,0.0,0.806,0.462,0.444
14,2025.2,-7,9,3,49,0.0,0.0,0.500,0.222,0.333
21,2025.2,0,9,9,230,9.0,3.0,0.500,0.222,1.000
22,2025.2,1,9,9,236,11.0,3.0,0.485,0.222,1.000
24,2025.2,3,9,9,223,10.0,3.0,0.500,0.222,1.000
28,2025.2,7,9,4,21,0.0,0.0,0.917,0.139,0.444
36,2026.1,-14,5,2,27,0.0,0.0,0.456,0.391,0.400
43,2026.1,-7,5,2,3,0.0,0.0,1.000,0.000,0.400
50,2026.1,0,5,5,64,14.0,4.0,0.500,0.214,1.000
51,2026.1,1,5,5,66,16.0,3.0,0.375,0.219,1.000


In [50]:
rolling_peak_df = (
    rolling_summary_df.loc[
        rolling_summary_df.groupby("Semestre")["commits"].idxmax()
    ][
        [
            "Semestre",
            "window_end_day",
            "commits",
            "equipes_ativas",
            "mediana_commits",
            "mediana_maior_participacao",
            "mediana_gini",
        ]
    ]
    .reset_index(drop=True)
)

team_peak_df = rolling_metrics_df.loc[
    rolling_metrics_df.groupby(TEAM_KEY)["commit_n"].idxmax()
][
    TEAM_KEY
    + [
        "window_end_day",
        "commit_n",
        "active_days",
        "author_n",
        "max_author_share",
        "author_gini",
    ]
].sort_values(TEAM_KEY)

rolling_checks = {
    "expected_team_day_rows": len(rolling_metrics_df) == 14 * 29,
    "seven_day_windows": (
        rolling_metrics_df["window_end"]
        - rolling_metrics_df["window_start"]
    ).eq(pd.Timedelta(days=7)).all(),
    "peak_2025_2_ends_at_p_plus_1": rolling_peak_df.set_index("Semestre").loc[
        "2025.2", "window_end_day"
    ]
    == 1,
    "peak_2026_1_ends_at_p_plus_1": rolling_peak_df.set_index("Semestre").loc[
        "2026.1", "window_end_day"
    ]
    == 1,
}
assert all(rolling_checks.values()), rolling_checks

display(pd.Series(rolling_checks, name="resultado"))
display(rolling_peak_df.round(3))
team_peak_df.round(3)

expected_team_day_rows          True
seven_day_windows               True
peak_2025_2_ends_at_p_plus_1    True
peak_2026_1_ends_at_p_plus_1    True
Name: resultado, dtype: bool

,Semestre,window_end_day,commits,equipes_ativas,mediana_commits,mediana_maior_participacao,mediana_gini
0,2025.2,1,236,9,11.0,0.485,0.222
1,2026.1,1,66,5,16.0,0.375,0.219


,Semestre,ID_Equipe,window_end_day,commit_n,active_days,author_n,max_author_share,author_gini
21,2025.2,TEAM_01,0,11,3,1,1.000,0.000
50,2025.2,TEAM_02,0,109,7,5,0.477,0.418
79,2025.2,TEAM_03,0,9,2,3,0.444,0.148
109,2025.2,TEAM_04,1,32,5,5,0.719,0.562
138,2025.2,TEAM_05,1,3,2,2,0.667,0.167
145,2025.2,TEAM_07,-21,30,5,4,0.500,0.433
182,2025.2,TEAM_08,-13,14,4,7,0.357,0.347
224,2025.2,TEAM_09,0,52,4,5,0.442,0.415
232,2025.2,TEAM_10,-21,8,2,3,0.750,0.417
282,2026.1,TEAM_01,0,1,1,1,1.000,0.000


In [51]:
rolling_figure = go.Figure()
for semester, group in rolling_summary_df.groupby("Semestre", sort=False):
    rolling_figure.add_trace(
        go.Scatter(
            x=group["window_end_day"],
            y=group["commits"],
            mode="lines+markers",
            name=f"Commits — {semester}",
        )
    )
rolling_figure.add_vline(
    x=0,
    line_dash="dash",
    annotation_text="Apresentação",
)
rolling_figure.update_layout(
    title="Atividade em janelas móveis de sete dias",
    xaxis_title="Dia final da janela relativo à apresentação",
    yaxis_title="Commits nas equipes do semestre",
    legend_title_text="Série",
)
display(HTML(rolling_figure.to_html(include_plotlyjs="cdn", full_html=False)))

concentration_rolling_figure = go.Figure()
for semester, group in rolling_summary_df.groupby("Semestre", sort=False):
    concentration_rolling_figure.add_trace(
        go.Scatter(
            x=group["window_end_day"],
            y=group["mediana_maior_participacao"],
            mode="lines+markers",
            name=semester,
        )
    )
concentration_rolling_figure.add_vline(
    x=0,
    line_dash="dash",
    annotation_text="Apresentação",
)
concentration_rolling_figure.update_layout(
    title="Concentração de autoria em janelas móveis de sete dias",
    xaxis_title="Dia final da janela relativo à apresentação",
    yaxis_title="Mediana da maior participação entre equipes ativas",
    yaxis_range=[0, 1],
    legend_title_text="Semestre",
)
display(
    HTML(
        concentration_rolling_figure.to_html(
            include_plotlyjs=False,
            full_html=False,
        )
    )
)

### Resultado das janelas sobrepostas

Nos dois semestres, o volume agregado máximo ocorre na janela de sete dias que termina em $P_i+1d$:

- 2025.2: 236 commits, com atividade nas 9 equipes;
- 2026.1: 66 commits, com atividade nas 5 equipes.

A janela que termina exatamente na apresentação contém 230 e 64 commits, respectivamente. Portanto, o pico agregado está concentrado ao redor da apresentação, não semanas antes.

A concentração de autoria não cresce de forma uniforme junto com o volume. Em 2025.2, a mediana da maior participação é 0,50 na janela que termina em $P_i$; em 2026.1, também é 0,50. Isso sugere um pico de atividade final disseminado entre equipes, mas não um aumento geral e monotônico de concentração em um único autor.

As trajetórias individuais são heterogêneas: 10 das 14 equipes atingem seu maior volume em janelas que terminam em $P_i$ ou $P_i+1d$; quatro atingem o pico antes. A análise móvel ajuda a localizar o pico, mas não produz observações independentes nem demonstra fricção.

### 6.2 Concentração estrutural versus final

M3c contextualiza M3b de duas formas:

1. **concentração estrutural:** histórico da branch padrão até $P_i+7d$;
2. **baseline sem sobreposição:** commits anteriores a $P_i-7d$ comparados à fase final.

A correlação estrutural–final é parcialmente mecânica porque a janela final integra o histórico completo. A comparação baseline–final é mais defensável para avaliar persistência ou mudança.

In [52]:
from scipy.stats import spearmanr


def concentration_for_interval(
    commits: pd.DataFrame,
    start: pd.Timestamp | None,
    end: pd.Timestamp,
) -> dict[str, float | int]:
    end_utc = end.tz_convert("UTC")
    current = commits[commits["committed_at"].lt(end_utc)]
    if start is not None:
        current = current[current["committed_at"].ge(start.tz_convert("UTC"))]
    counts = current["author_key"].value_counts()
    shares = counts / len(current) if len(current) else pd.Series(dtype=float)
    return {
        "commit_n": int(len(current)),
        "author_n": int(len(counts)),
        "max_author_share": float(shares.max()) if len(shares) else np.nan,
        "author_gini": gini(counts) if len(counts) else np.nan,
    }


concentration_scope_rows = []
for anchor in presentation_anchors_df.to_dict("records"):
    presentation = anchor["presentation_anchor"]
    final_start = presentation - pd.Timedelta(days=7)
    final_end = presentation + pd.Timedelta(days=7)
    git_dir = PARENT_CACHE_DIR / f"{anchor['repository']}.git"
    commits = parent_commit_identities(
        git_dir,
        pd.Timestamp("1970-01-01", tz="UTC"),
        final_end,
    )
    row = {
        "Semestre": anchor["Semestre"],
        "ID_Equipe": anchor["ID_Equipe"],
    }
    for scope, start, end in (
        ("baseline", None, final_start),
        ("final", final_start, final_end),
        ("whole", None, final_end),
    ):
        metrics = concentration_for_interval(commits, start, end)
        row.update({f"{scope}_{name}": value for name, value in metrics.items()})
    row["delta_final_vs_baseline_max_share"] = (
        row["final_max_author_share"] - row["baseline_max_author_share"]
    )
    row["delta_final_vs_baseline_gini"] = (
        row["final_author_gini"] - row["baseline_author_gini"]
    )
    concentration_scope_rows.append(row)

concentration_scope_df = pd.DataFrame(concentration_scope_rows)
concentration_scope_df.sort_values(TEAM_KEY).round(3)

,Semestre,ID_Equipe,baseline_commit_n,baseline_author_n,baseline_max_author_share,baseline_author_gini,final_commit_n,final_author_n,final_max_author_share,final_author_gini,whole_commit_n,whole_author_n,whole_max_author_share,whole_author_gini,delta_final_vs_baseline_max_share,delta_final_vs_baseline_gini
0,2025.2,TEAM_01,8,2,0.500,0.000,11,1,1.000,0.000,19,2,0.789,0.289,0.500,0.000
1,2025.2,TEAM_02,92,6,0.489,0.547,109,5,0.477,0.418,201,6,0.483,0.530,-0.012,-0.129
2,2025.2,TEAM_03,0,0,NaN,NaN,9,3,0.444,0.148,9,3,0.444,0.148,NaN,NaN
3,2025.2,TEAM_04,24,5,0.458,0.450,39,5,0.641,0.554,63,5,0.524,0.476,0.183,0.104
4,2025.2,TEAM_05,0,0,NaN,NaN,3,2,0.667,0.167,3,2,0.667,0.167,NaN,NaN
5,2025.2,TEAM_07,67,6,0.448,0.550,7,3,0.429,0.095,74,6,0.446,0.523,-0.019,-0.455
6,2025.2,TEAM_08,46,8,0.283,0.332,14,4,0.357,0.250,60,8,0.217,0.325,0.075,-0.082
7,2025.2,TEAM_09,55,5,0.691,0.538,52,5,0.442,0.415,107,5,0.570,0.475,-0.249,-0.123
8,2025.2,TEAM_10,11,4,0.545,0.341,7,3,0.429,0.190,18,5,0.500,0.444,-0.117,-0.150
9,2026.1,TEAM_01,2,2,0.500,0.000,1,1,1.000,0.000,3,2,0.667,0.167,0.500,0.000


In [53]:
concentration_relation_rows = []
for metric in ("max_author_share", "author_gini"):
    for reference in ("whole", "baseline"):
        columns = [f"{reference}_{metric}", f"final_{metric}"]
        valid = concentration_scope_df[columns].dropna()
        rho, p_value = spearmanr(valid.iloc[:, 0], valid.iloc[:, 1])
        concentration_relation_rows.append(
            {
                "reference": reference,
                "metric": metric,
                "n": len(valid),
                "rho": rho,
                "p_value": p_value,
                "overlapping_with_final": reference == "whole",
            }
        )
concentration_relation_df = pd.DataFrame(concentration_relation_rows)

concentration_change_summary_df = (
    concentration_scope_df.groupby("Semestre", as_index=False)
    .agg(
        equipes=("ID_Equipe", "size"),
        mediana_delta_max_share=("delta_final_vs_baseline_max_share", "median"),
        mediana_delta_gini=("delta_final_vs_baseline_gini", "median"),
        equipes_aumento_max_share=("delta_final_vs_baseline_max_share", lambda values: values.gt(0).sum()),
        equipes_reducao_max_share=("delta_final_vs_baseline_max_share", lambda values: values.lt(0).sum()),
    )
)

concentration_relation_checks = {
    "whole_final_max_share_relation_is_positive": concentration_relation_df.query(
        "reference == 'whole' and metric == 'max_author_share'"
    )["rho"].iloc[0]
    > 0,
    "baseline_final_comparison_is_non_overlapping": not concentration_relation_df.query(
        "reference == 'baseline'"
    )["overlapping_with_final"].any(),
    "baseline_missing_for_two_late_starting_teams": concentration_scope_df[
        "baseline_max_author_share"
    ].isna().sum()
    == 2,
}
assert all(concentration_relation_checks.values()), concentration_relation_checks

display(concentration_relation_df.round(3))
concentration_change_summary_df.round(3)

,reference,metric,n,rho,p_value,overlapping_with_final
0,whole,max_author_share,14,0.773,0.001,True
1,baseline,max_author_share,12,0.612,0.035,False
2,whole,author_gini,14,0.642,0.013,True
3,baseline,author_gini,12,0.607,0.036,False


,Semestre,equipes,mediana_delta_max_share,mediana_delta_gini,equipes_aumento_max_share,equipes_reducao_max_share
0,2025.2,9,-0.012,-0.123,3,4
1,2026.1,5,0.011,-0.212,3,2


### Relação entre M3b e M3c

A concentração acumulada e a concentração final estão positivamente associadas, mas a versão acumulada inclui a própria janela final. Essa correlação é descritiva e parcialmente mecânica.

A comparação sem sobreposição, baseline anterior versus fase final, ainda mostra associação moderada. Isso sugere persistência: parte da concentração final já caracteriza a equipe antes da fase final.

Entretanto, a mediana da mudança da maior participação é próxima de zero em ambos os semestres. Portanto, os dados não sustentam aumento geral de concentração na fase final; sustentam heterogeneidade entre equipes e persistência parcial do padrão de autoria.

## 7. Triagem de variações e combinações

As candidatas abaixo são avaliadas pela informação adicional em relação a M3a (`final_window_share`):

- continuidade pós-apresentação;
- proporção dos dias ativos na fase final;
- intensidade relativa por dia ativo;
- fração histórica atribuível ao autor dominante final;
- participação e momento da semana móvel de pico;
- aceleração entre a penúltima e a última semana pré-apresentação.

Correlação alta com M3a indica provável redundância; correlação baixa não garante validade, mas sugere dimensão distinta.

In [58]:
candidate_rows = []
rolling_peak_by_team_df = rolling_metrics_df.loc[
    rolling_metrics_df.groupby(TEAM_KEY)["commit_n"].idxmax()
][TEAM_KEY + ["commit_n", "window_end_day"]].rename(
    columns={
        "commit_n": "peak_7d_commit_n",
        "window_end_day": "peak_7d_end_day",
    }
)

for anchor in presentation_anchors_df.to_dict("records"):
    git_dir = PARENT_CACHE_DIR / f"{anchor['repository']}.git"
    presentation = anchor["presentation_anchor"]
    denominator_end = presentation + pd.Timedelta(days=7)
    commits = parent_commit_identities(
        git_dir,
        pd.Timestamp("1970-01-01", tz="UTC"),
        denominator_end,
    )
    total_active_days = (
        commits["committed_at"]
        .dt.tz_convert("America/Fortaleza")
        .dt.date.nunique()
        if len(commits)
        else 0
    )
    candidate_rows.append(
        {
            "Semestre": anchor["Semestre"],
            "ID_Equipe": anchor["ID_Equipe"],
            "total_active_days": int(total_active_days),
        }
    )

candidate_metrics_df = (
    final_commit_share_df[
        TEAM_KEY
        + [
            "total_commit_n",
            "final_window_commit_n",
            "final_window_share",
            "post_share",
        ]
    ]
    .merge(pd.DataFrame(candidate_rows), on=TEAM_KEY, validate="one_to_one")
    .merge(
        final_phase_metrics_df[
            final_phase_metrics_df["phase"].eq("pre")
        ][TEAM_KEY + ["active_days", "max_author_share"]].rename(
            columns={
                "active_days": "final_pre_active_days",
                "max_author_share": "final_pre_max_author_share",
            }
        ),
        on=TEAM_KEY,
        validate="one_to_one",
    )
    .merge(
        final_phase_metrics_df[
            final_phase_metrics_df["phase"].eq("post")
        ][TEAM_KEY + ["active_days"]].rename(
            columns={"active_days": "final_post_active_days"}
        ),
        on=TEAM_KEY,
        validate="one_to_one",
    )
    .merge(rolling_peak_by_team_df, on=TEAM_KEY, validate="one_to_one")
)
candidate_metrics_df["final_active_days"] = (
    candidate_metrics_df["final_pre_active_days"]
    + candidate_metrics_df["final_post_active_days"]
)
candidate_metrics_df["final_active_day_share"] = (
    candidate_metrics_df["final_active_days"]
    / candidate_metrics_df["total_active_days"]
)
candidate_metrics_df["commit_intensity_ratio"] = (
    candidate_metrics_df["final_window_share"]
    / candidate_metrics_df["final_active_day_share"]
)
candidate_metrics_df["dominant_final_author_repository_share"] = (
    candidate_metrics_df["final_window_share"]
    * concentration_scope_df.set_index(TEAM_KEY)["final_max_author_share"]
    .reindex(pd.MultiIndex.from_frame(candidate_metrics_df[TEAM_KEY]))
    .to_numpy()
)
candidate_metrics_df["peak_7d_share"] = (
    candidate_metrics_df["peak_7d_commit_n"]
    / candidate_metrics_df["total_commit_n"]
)

prior_week = rolling_metrics_df[
    rolling_metrics_df["window_end_day"].eq(-7)
][TEAM_KEY + ["commit_n"]].rename(columns={"commit_n": "prior_week_commit_n"})
final_pre_week = rolling_metrics_df[
    rolling_metrics_df["window_end_day"].eq(0)
][TEAM_KEY + ["commit_n"]].rename(columns={"commit_n": "final_pre_week_commit_n"})
candidate_metrics_df = candidate_metrics_df.merge(
    prior_week,
    on=TEAM_KEY,
    validate="one_to_one",
).merge(
    final_pre_week,
    on=TEAM_KEY,
    validate="one_to_one",
)
candidate_metrics_df["pre_acceleration_contrast"] = (
    candidate_metrics_df["final_pre_week_commit_n"]
    - candidate_metrics_df["prior_week_commit_n"]
) / (
    candidate_metrics_df["final_pre_week_commit_n"]
    + candidate_metrics_df["prior_week_commit_n"]
)

candidate_metrics_df.sort_values(TEAM_KEY).round(3)

,Semestre,ID_Equipe,total_commit_n,final_window_commit_n,final_window_share,post_share,total_active_days,final_pre_active_days,final_pre_max_author_share,final_post_active_days,peak_7d_commit_n,peak_7d_end_day,final_active_days,final_active_day_share,commit_intensity_ratio,dominant_final_author_repository_share,peak_7d_share,prior_week_commit_n,final_pre_week_commit_n,pre_acceleration_contrast
0,2025.2,TEAM_01,19,11,0.579,0.000,8,3,1.000,0,11,0,3,0.375,1.544,0.579,0.579,0,11,1.000
1,2025.2,TEAM_02,201,109,0.542,0.000,29,7,0.477,0,109,0,7,0.241,2.247,0.259,0.542,36,109,0.503
2,2025.2,TEAM_03,9,9,1.000,0.000,2,2,0.444,0,9,0,2,1.000,1.000,0.444,1.000,0,9,1.000
3,2025.2,TEAM_04,63,39,0.619,0.190,17,4,0.704,3,32,1,7,0.412,1.503,0.397,0.508,3,27,0.800
4,2025.2,TEAM_05,3,3,1.000,0.667,2,1,1.000,1,3,1,2,1.000,1.000,0.667,1.000,0,1,1.000
5,2025.2,TEAM_07,74,7,0.095,0.014,16,3,0.500,1,30,-21,4,0.250,0.378,0.041,0.405,0,6,1.000
6,2025.2,TEAM_08,60,14,0.233,0.100,21,5,0.500,1,14,-13,6,0.286,0.817,0.083,0.233,10,8,-0.111
7,2025.2,TEAM_09,107,52,0.486,0.000,18,4,0.442,0,52,0,4,0.222,2.187,0.215,0.486,0,52,1.000
8,2025.2,TEAM_10,18,7,0.389,0.000,6,3,0.429,0,8,-21,3,0.500,0.778,0.167,0.444,0,7,1.000
9,2026.1,TEAM_01,3,1,0.333,0.000,3,1,1.000,0,1,0,1,0.333,1.000,0.333,0.333,0,1,1.000


In [59]:
CANDIDATE_COLUMNS = [
    "post_share",
    "final_active_day_share",
    "commit_intensity_ratio",
    "final_pre_max_author_share",
    "dominant_final_author_repository_share",
    "peak_7d_share",
    "peak_7d_end_day",
    "pre_acceleration_contrast",
]

candidate_redundancy_rows = []
for candidate in CANDIDATE_COLUMNS:
    valid = candidate_metrics_df[["final_window_share", candidate]].dropna()
    rho, p_value = spearmanr(valid["final_window_share"], valid[candidate])
    candidate_redundancy_rows.append(
        {
            "candidate": candidate,
            "n": len(valid),
            "rho_with_m3a": rho,
            "p_value": p_value,
            "absolute_rho": abs(rho),
        }
    )
candidate_redundancy_df = pd.DataFrame(candidate_redundancy_rows).sort_values(
    "absolute_rho"
)

candidate_semester_summary_df = (
    candidate_metrics_df.groupby("Semestre", as_index=False)
    .agg(
        mediana_post_share=("post_share", "median"),
        mediana_final_active_day_share=("final_active_day_share", "median"),
        mediana_commit_intensity_ratio=("commit_intensity_ratio", "median"),
        mediana_dominant_final_author_share=("dominant_final_author_repository_share", "median"),
        mediana_peak_7d_share=("peak_7d_share", "median"),
        mediana_acceleration=("pre_acceleration_contrast", "median"),
        equipes_aceleraram=("pre_acceleration_contrast", lambda values: values.gt(0).sum()),
        equipes_desaceleraram=("pre_acceleration_contrast", lambda values: values.lt(0).sum()),
    )
)

display(candidate_redundancy_df.round(3))
candidate_semester_summary_df.round(3)

,candidate,n,rho_with_m3a,p_value,absolute_rho
0,post_share,14,0.101,0.731,0.101
7,pre_acceleration_contrast,14,0.254,0.381,0.254
3,final_pre_max_author_share,14,0.353,0.216,0.353
2,commit_intensity_ratio,14,0.387,0.172,0.387
6,peak_7d_end_day,14,0.587,0.027,0.587
1,final_active_day_share,14,0.783,0.001,0.783
5,peak_7d_share,14,0.921,0.000,0.921
4,dominant_final_author_repository_share,14,0.922,0.000,0.922


,Semestre,mediana_post_share,mediana_final_active_day_share,mediana_commit_intensity_ratio,mediana_dominant_final_author_share,mediana_peak_7d_share,mediana_acceleration,equipes_aceleraram,equipes_desaceleraram
0,2025.2,0.000,0.375,1.000,0.259,0.508,1.0,8,1
1,2026.1,0.014,0.217,1.259,0.085,0.286,1.0,5,0


### Decisão após a triagem

**Manter como saídas ou componentes:**

- **continuidade pós-apresentação (`post_share`):** quase independente de M3a ($\rho=0{,}101$); distingue equipes que continuaram alterando o repositório. Deve ser acompanhada do status de autorização, quando conhecido;
- **momento da semana de pico (`peak_7d_end_day`):** acrescenta localização temporal ($\rho=0{,}587$ com M3a); permanece dentro de M3c;
- **concentração pré-final (`final_pre_max_author_share`):** acrescenta dimensão de autoria ($\rho=0{,}353$); permanece em M3b;
- **dias ativos finais:** reportar como denominador operacional, não como nova métrica independente.

**Manter apenas como diagnóstico exploratório:**

- **razão de intensidade por dia ativo:** correlação moderada com M3a ($\rho=0{,}387$), mas interpretação menos direta e sensível a poucos dias ativos;
- **aceleração entre semanas:** baixa redundância ($\rho=0{,}254$), porém 13/14 equipes aceleram e o índice satura em 1 quando a semana anterior tem zero commits.

**Descartar como novas métricas:**

- **participação da semana móvel de pico:** redundante com M3a ($\rho=0{,}921$);
- **participação histórica do autor dominante final:** combinação redundante de M3a e M3b ($\rho=0{,}922$);
- **participação dos dias ativos finais:** fortemente associada a M3a ($\rho=0{,}783$); usar apenas como contexto.

Assim, a extensão útil é pequena: M3a responde **quando**, M3b responde **quem**, M3c responde **como o padrão evolui**, e `post_share` explicita continuidade após a apresentação.

In [60]:
candidate_lookup = candidate_redundancy_df.set_index("candidate")
candidate_decision_checks = {
    "post_share_is_distinct_from_m3a": round(float(candidate_lookup.loc["post_share", "rho_with_m3a"]), 3) == 0.101,
    "peak_timing_adds_information": round(float(candidate_lookup.loc["peak_7d_end_day", "rho_with_m3a"]), 3) == 0.587,
    "author_concentration_adds_information": round(float(candidate_lookup.loc["final_pre_max_author_share", "rho_with_m3a"]), 3) == 0.353,
    "peak_share_is_redundant": candidate_lookup.loc["peak_7d_share", "absolute_rho"] > 0.9,
    "dominant_author_composite_is_redundant": candidate_lookup.loc[
        "dominant_final_author_repository_share", "absolute_rho"
    ]
    > 0.9,
    "acceleration_saturates": candidate_metrics_df[
        "pre_acceleration_contrast"
    ].gt(0).sum()
    == 13,
}
assert all(candidate_decision_checks.values()), candidate_decision_checks
pd.Series(candidate_decision_checks, name="resultado")

post_share_is_distinct_from_m3a           True
peak_timing_adds_information              True
author_concentration_adds_information     True
peak_share_is_redundant                   True
dominant_author_composite_is_redundant    True
acceleration_saturates                    True
Name: resultado, dtype: bool

## 8. Síntese e limitações

### 8.1 Interpretação da janela final

A janela pré-apresentação tem cobertura completa: 14/14 equipes. A fase pós-apresentação possui commits em 7/14 equipes.

Isso não significa que sete equipes tinham autorização formal para continuar. O notebook conhece autorização apenas para TEAM_04/2025.2; para as demais, registra-se somente atividade pós-âncora.

A separação pré/pós é obrigatória. A janela combinada descreve atividade tardia na fase final, mas não deve ocultar em qual lado da apresentação os commits ocorreram.

Contagens, proporções, dias ativos e concentração são os resultados principais. Churn é contexto complementar porque arquivos gerados e binários podem dominar seu volume.

In [36]:
phase_figure = go.Figure()
for phase, label in (("pre", "7 dias antes"), ("post", "7 dias depois")):
    current = final_phase_metrics_df[final_phase_metrics_df["phase"].eq(phase)].copy()
    current["team_semester"] = current["Semestre"] + "/" + current["ID_Equipe"]
    phase_figure.add_trace(
        go.Bar(
            x=current["team_semester"],
            y=current["commit_n"],
            name=label,
            customdata=current[["active_days", "author_n", "max_author_share"]],
            hovertemplate=(
                "%{x}<br>Commits: %{y}<br>Dias ativos: %{customdata[0]}"
                "<br>Autores: %{customdata[1]}<br>Maior participação: %{customdata[2]:.2f}"
                "<extra></extra>"
            ),
        )
    )
phase_figure.update_layout(
    title="Atividade tardia ao redor da apresentação",
    xaxis_title="Equipe-semestre",
    yaxis_title="Commits na branch padrão do repositório original",
    barmode="group",
)
display(HTML(phase_figure.to_html(include_plotlyjs="cdn", full_html=False)))

### 8.2 Baixa atividade não implica bom planejamento

Poucos commits podem representar estabilidade, mas também baixa produção, commits grandes, trabalho fora do Git ou atraso. O número depende da prática de versionamento.

“Poucos commits” é **baixa atividade relativa**, não bom planejamento. Neste conjunto, não existe outlier inferior formal pelo critério de Tukey; existem apenas equipes nos menores postos relativos.

In [25]:
t3_relative_activity_df = recalculated_cut_df[
    recalculated_cut_df["cut"].eq("T3")
][TEAM_KEY + ["commit_n", "churn", "author_n", "max_author_share"]].copy()

semester_groups = t3_relative_activity_df.groupby("Semestre")["commit_n"]
t3_relative_activity_df["rank_baixa_atividade"] = semester_groups.rank(
    method="min",
    ascending=True,
)
t3_relative_activity_df["participacao_nos_commits_do_semestre"] = (
    t3_relative_activity_df["commit_n"]
    / semester_groups.transform("sum")
)
q1 = semester_groups.transform(lambda values: values.quantile(0.25))
q3 = semester_groups.transform(lambda values: values.quantile(0.75))
t3_relative_activity_df["limite_inferior_tukey"] = q1 - 1.5 * (q3 - q1)
t3_relative_activity_df["outlier_inferior_formal"] = t3_relative_activity_df[
    "commit_n"
].lt(t3_relative_activity_df["limite_inferior_tukey"])

t3_relative_activity_df.sort_values(
    ["Semestre", "rank_baixa_atividade"]
).round(3)

,Semestre,ID_Equipe,commit_n,churn,author_n,max_author_share,rank_baixa_atividade,participacao_nos_commits_do_semestre,limite_inferior_tukey,outlier_inferior_formal
26,2025.2,TEAM_05,1,7110.0,1,1.000,1.0,0.003,-47.0,False
35,2025.2,TEAM_08,3,304.0,3,0.333,2.0,0.009,-47.0,False
41,2025.2,TEAM_10,7,2202.0,3,0.429,3.0,0.020,-47.0,False
14,2025.2,TEAM_03,9,21198.0,3,0.444,4.0,0.026,-47.0,False
2,2025.2,TEAM_01,12,1366581.0,1,1.000,5.0,0.034,-47.0,False
32,2025.2,TEAM_07,36,11762127.0,5,0.500,6.0,0.102,-47.0,False
20,2025.2,TEAM_04,43,25088.0,5,0.581,7.0,0.122,-47.0,False
38,2025.2,TEAM_09,77,134869.0,5,0.558,8.0,0.219,-47.0,False
8,2025.2,TEAM_02,164,53210.0,5,0.500,9.0,0.466,-47.0,False
5,2026.1,TEAM_01,1,7213.0,1,1.000,1.0,0.008,-14.5,False


## 9. Tabelas e figuras para o paper

Saídas recomendadas:

- tabela M3a com numerador, denominador, `pre_share`, `post_share` e participação final;
- tabela M3b com commits, dias ativos, autores, maior participação e Gini por fase;
- figura de participação pré/pós por equipe;
- figura M3c de atividade e concentração em janelas móveis;
- tabela de baseline versus fase final.

Equipes sem commits permanecem no denominador de cobertura, mas não recebem concentração igual a zero.

In [13]:
paper_table_df = cut_summary_df[
    [
        "Semestre",
        "cut",
        "equipes",
        "equipes_ativas",
        "commits",
        "mediana_autores_ativos",
        "mediana_maior_participacao",
        "mediana_gini",
    ]
].copy()

window_figure = go.Figure(
    data=[
        go.Bar(
            x=baseline_comparison_df["definição"],
            y=baseline_comparison_df["equipes_ativas"],
            text=baseline_comparison_df["equipes_ativas"],
            name="Equipes ativas",
        )
    ]
)
window_figure.update_layout(
    title="Cobertura: T3 completo versus janela pré-T3 de 72 horas",
    xaxis_title="Definição",
    yaxis_title="Equipes com commits",
    yaxis_range=[0, 14],
)

display(paper_table_df.round(3))
display(HTML(window_figure.to_html(include_plotlyjs="cdn", full_html=False)))

,Semestre,cut,equipes,equipes_ativas,commits,mediana_autores_ativos,mediana_maior_participacao,mediana_gini
0,2025.2,T1,9,4,24,2.0,0.619,0.071
1,2025.2,T2,9,6,141,5.5,0.475,0.404
2,2025.2,T3,9,9,352,3.0,0.500,0.190
3,2026.1,T1,5,5,87,3.0,0.636,0.364
4,2026.1,T2,5,4,88,6.5,0.393,0.375
5,2026.1,T3,5,5,120,4.0,0.447,0.221


## 10. Checagens de consistência e regressão

As checagens bloqueiam:

- mudança de RQ;
- alteração silenciosa do baseline;
- mistura de janelas;
- concentração preenchida quando não há atividade;
- vazamento de identidade de autores;
- divergência dos valores principais M3a–M3c;
- perda das 14 âncoras ou mirrors.

In [56]:
single_author_rows = recalculated_cut_df[
    recalculated_cut_df["author_n"].eq(1)
]
no_activity_rows = recalculated_cut_df[
    recalculated_cut_df["commit_n"].eq(0)
]
parent_mirrors = list(PARENT_CACHE_DIR.glob("*.git"))
pre_phase = final_phase_metrics_df[final_phase_metrics_df["phase"].eq("pre")]
post_phase = final_phase_metrics_df[final_phase_metrics_df["phase"].eq("post")]
authorized_rows = post_phase[post_phase["authorized_extension"]]
summary_by_semester = final_commit_share_summary_df.set_index("Semestre")
relation_lookup = concentration_relation_df.set_index(["reference", "metric"])

regression_checks = {
    "rq_link_is_rq2": detected_rq == "RQ2",
    "primary_metric_is_final_commit_share": CONFIG["primary_metric"] == "final_commit_share",
    "m3c_is_registered": "rolling_author_concentration_7d" in CONFIG["secondary_metrics"],
    "published_full_cut_reproduced": max(max_errors.values()) < CONFIG["regression_tolerance"],
    "published_t3_has_14_active_teams": len(full_t3) == 14,
    "legacy_pre_t3_72h_has_7_active_teams": len(active_72h) == 7,
    "no_activity_has_missing_concentration": no_activity_rows[["max_author_share", "author_share_median", "author_gini"]].isna().all().all(),
    "single_author_max_share_is_one": single_author_rows["max_author_share"].eq(1).all(),
    "single_author_gini_is_zero": single_author_rows["author_gini"].eq(0).all(),
    "fourteen_vote_anchors": len(presentation_anchors_df) == 14,
    "fourteen_parent_mirrors": len(parent_mirrors) == 14,
    "symmetric_seven_day_exposure": final_phase_metrics_df["exposure_days"].eq(7).all(),
    "all_teams_active_pre_presentation": pre_phase["commit_n"].gt(0).all(),
    "seven_teams_active_post_presentation": post_phase["commit_n"].gt(0).sum() == 7,
    "only_declared_extension_is_team04_2025_2": authorized_rows[TEAM_KEY].to_dict("records") == [{"Semestre": "2025.2", "ID_Equipe": "TEAM_04"}],
    "public_metrics_contain_no_author_identity": not {"author_key", "author_email", "author_name"}.intersection(final_phase_metrics_df.columns),
    "fourteen_final_commit_shares": len(final_commit_share_df) == 14,
    "share_components_add_exactly": share_checks["shares_add_exactly"],
    "median_share_2025_2_is_0_542": np.isclose(summary_by_semester.loc["2025.2", "mediana_participacao_final"], 0.5422885572139303),
    "median_share_2026_1_is_0_286": np.isclose(summary_by_semester.loc["2026.1", "mediana_participacao_final"], 0.2857142857142857),
    "pooled_median_share_is_0_437": np.isclose(summary_by_semester.loc["Todos", "mediana_participacao_final"], 0.4374350986500519),
    "pooled_weighted_share_is_0_379": np.isclose(summary_by_semester.loc["Todos", "participacao_ponderada_final"], 322 / 849),
    "three_small_denominators_flagged": final_commit_share_df["small_denominator_lt_10"].sum() == 3,
    "rolling_rows_complete": rolling_checks["expected_team_day_rows"],
    "rolling_peak_is_p_plus_1_both_semesters": rolling_checks["peak_2025_2_ends_at_p_plus_1"] and rolling_checks["peak_2026_1_ends_at_p_plus_1"],
    "whole_final_max_share_rho_is_0_773": round(float(relation_lookup.loc[("whole", "max_author_share"), "rho"]), 3) == 0.773,
    "baseline_final_max_share_rho_is_0_612": round(float(relation_lookup.loc[("baseline", "max_author_share"), "rho"]), 3) == 0.612,
    "whole_final_gini_rho_is_0_642": round(float(relation_lookup.loc[("whole", "author_gini"), "rho"]), 3) == 0.642,
    "baseline_final_gini_rho_is_0_607": round(float(relation_lookup.loc[("baseline", "author_gini"), "rho"]), 3) == 0.607,
    "two_teams_without_pre_final_baseline": concentration_scope_df["baseline_max_author_share"].isna().sum() == 2,
    "t3_is_not_final_deadline": CONFIG["t3_is_final_deadline"] is False,
    "no_formal_lower_commit_outliers": not t3_relative_activity_df["outlier_inferior_formal"].any(),
    "no_llm_calls_required": CONFIG["llm_calls_required"] is False,
}
assert all(regression_checks.values()), regression_checks
pd.Series(regression_checks, name="resultado")

rq_link_is_rq2                               True
primary_metric_is_final_commit_share         True
m3c_is_registered                            True
published_full_cut_reproduced                True
published_t3_has_14_active_teams             True
legacy_pre_t3_72h_has_7_active_teams         True
no_activity_has_missing_concentration        True
single_author_max_share_is_one               True
single_author_gini_is_zero                   True
fourteen_vote_anchors                        True
fourteen_parent_mirrors                      True
symmetric_seven_day_exposure                 True
all_teams_active_pre_presentation            True
seven_teams_active_post_presentation         True
only_declared_extension_is_team04_2025_2     True
public_metrics_contain_no_author_identity    True
fourteen_final_commit_shares                 True
share_components_add_exactly                 True
median_share_2025_2_is_0_542                 True
median_share_2026_1_is_0_286                 True


## 11. Decisão de refatoração

Substituir o M3 ambíguo por três saídas complementares:

1. **M3a — Participação dos commits na fase final:** proporção dos commits acumulados até $P_i+7d$ ocorrida em $[P_i-7d,P_i+7d)$, com parcelas pré/pós. Resultado principal.
2. **M3b — Concentração de autoria na fase final:** autores ativos, maior participação e Gini nas fases `pre` e `post`. Contexto de autoria.
3. **M3c — Dinâmica da concentração e atividade:** concentração estrutural, baseline anterior e trajetória móvel. Contexto temporal.

A trajetória legada T1–T3 e a janela de 72 horas antes do início agregado de T3 permanecem como baseline ou sensibilidade.

Regras:

- último voto T3 como $P_i$;
- branch padrão do mirror read-only do repositório original;
- `committer date` para inclusão temporal;
- denominador M3a encerrado em $P_i+7d$;
- numerador e denominador sempre visíveis;
- `pre_share` e `post_share` separados;
- denominadores pequenos sinalizados;
- janelas móveis tratadas como autocorrelacionadas;
- baseline e fase final sem sobreposição para avaliar mudança;
- extensões autorizadas registradas, nunca inferidas;
- nenhuma interpretação como uso de IA, esforço, produtividade, fricção, rework ou qualidade de planejamento.

M3 continua sendo contexto da RQ2. O contraste principal permanece M4 versus M5.

## 12. Decisão sobre a pipeline

A refatoração é **determinística e não requer chamadas LLM**.

Ordem recomendada:

1. preservar o CSV legado;
2. persistir equipe→último voto T3 com timezone;
3. persistir URL, branch padrão, HEAD e hash dos mirrors pais;
4. manter `repos_parent_cache/` local e ignorado pelo Git;
5. gerar M3a, M3b e M3c;
6. preservar fases `pre` e `post`;
7. marcar TEAM_04/2025.2 como extensão autorizada declarada;
8. remover o prefixo `ai_*` do novo contrato;
9. atualizar tabelas e resultados da RQ2.

Não é necessário regenerar áudio, NLP ou survey. A extração Git deve ser refeita dos mirrors pais, sem qualquer push.

## 13. Correções necessárias no paper_v8

1. Substituir “Pre-Deadline Author-Concentration Density” por M3a–M3c.
2. Definir $P_i$ como último voto T3 da equipe.
3. Documentar janelas pré e pós de sete dias.
4. Usar commits da branch padrão acumulados até $P_i+7d$ no denominador.
5. Declarar mirror read-only do repositório original e `committer date`.
6. Reportar M3a: medianas 54,2% (2025.2), 28,6% (2026.1), 43,7% geral; 37,9% ponderada.
7. Reportar numeradores 251/554 e 71/295.
8. Sinalizar três denominadores menores que dez.
9. Reportar M3b sem inferir esforço ou fricção.
10. Reportar M3c: pico móvel em $P+1d$ nos dois semestres e 10/14 picos individuais em $P$ ou $P+1d$.
11. Qualificar correlação estrutural–final como parcialmente mecânica ($\rho=0{,}773$); baseline–final sem sobreposição é $\rho=0{,}612$.
12. Não afirmar aumento geral de concentração: medianas da mudança ficam próximas de zero.
13. Manter parcelas pré/pós visíveis e registrar TEAM_04/2025.2 separadamente.
14. Usar “atividade tardia na fase final”, não “deadline crunch”.
15. Manter M3 como contexto; M4 versus M5 continua central à RQ2.
16. Remover nomenclatura `ai_*` dos novos artefatos.
17. Registrar ameaças: autoria não mede esforço/coordenação; último voto é proxy; janelas móveis são autocorrelacionadas; churn pode ser dominado por gerados/binários.

## 14. Exportação e evidências de reprodução

O pacote reproduzível inclui:

- configuração e vínculo M3→RQ2;
- hashes dos inputs e HEADs dos mirrors;
- M3a por equipe e resumo;
- M3b por equipe/fase;
- M3c estrutural e móvel;
- matriz de redundância das variantes;
- baseline legado;
- checagens de regressão.

A escrita permanece desativada por padrão (`export_outputs=False`).

In [61]:
def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_hashes = {
    str(path.relative_to(PROJECT_ROOT)): file_sha256(path)
    for path in (PAPER_PATH, GIT_COMMITS_PATH, SOURCE_PATH, M3_PATH)
}
parent_heads = {
    mirror.name: subprocess.check_output(
        ["git", f"--git-dir={mirror}", "rev-parse", "HEAD"],
        text=True,
    ).strip()
    for mirror in sorted(PARENT_CACHE_DIR.glob("*.git"))
}
evidence_manifest = {
    "metric": CONFIG["metric"],
    "rq_validation": validation_log,
    "config": CONFIG,
    "input_sha256": input_hashes,
    "parent_mirror_heads": parent_heads,
    "regression_checks": regression_checks,
    "candidate_decision_checks": candidate_decision_checks,
    "outputs": [
        "m3a_final_commit_share_by_team.csv",
        "m3a_final_commit_share_summary.csv",
        "m3b_author_concentration_by_team_phase.csv",
        "m3c_structural_vs_final_concentration.csv",
        "m3c_rolling_concentration_7d.csv",
        "m3c_rolling_concentration_summary.csv",
        "m3_candidate_redundancy.csv",
        "m3_legacy_author_concentration_by_cut.csv",
        "m3_evidence_manifest.json",
    ],
}

if CONFIG["export_outputs"]:
    output_dir = PROJECT_ROOT / "paper_v9/data/m3_refactor"
    output_dir.mkdir(parents=True, exist_ok=True)
    final_commit_share_df.to_csv(output_dir / evidence_manifest["outputs"][0], index=False)
    final_commit_share_summary_df.to_csv(output_dir / evidence_manifest["outputs"][1], index=False)
    final_phase_metrics_df.to_csv(output_dir / evidence_manifest["outputs"][2], index=False)
    concentration_scope_df.to_csv(output_dir / evidence_manifest["outputs"][3], index=False)
    rolling_metrics_df.to_csv(output_dir / evidence_manifest["outputs"][4], index=False)
    rolling_summary_df.to_csv(output_dir / evidence_manifest["outputs"][5], index=False)
    candidate_redundancy_df.to_csv(output_dir / evidence_manifest["outputs"][6], index=False)
    recalculated_cut_df.to_csv(output_dir / evidence_manifest["outputs"][7], index=False)
    (output_dir / evidence_manifest["outputs"][8]).write_text(
        json.dumps(evidence_manifest, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

evidence_manifest

{'metric': 'M3',
 'rq_validation': {'metric': 'M3',
  'expected_rq': 'RQ2',
  'detected_rq': 'RQ2',
  'rq_text': 'How does the temporal density of repository activity contrast with the qualitative typification of human coordination friction across the project lifecycle?',
  'm3_section_excerpt': "\\textbf{M3 -- Pre-Deadline Author-Concentration Density.} For team $i$ in\nsemester $s$, we take every commit in the 72-hour window immediately\npreceding that team's $T_3$ checkpoint and group it by the committing\nteammate's identifier. Let $C(a,i,s)$ be teammate $a$'s commit count in that\nwindow and $C(i,s)$ the window's total commit count; we report the median\nper-teammate commit share and the Gini coefficient of the per-teammate\ncommit-count distribution:\n\\[\n\\begin{aligned}\n\\mathrm{AuthorShare}(i,s)\n  &= \\underset{a}{\\mathrm{median}}\n     \\left(\\frac{C(a,i,s)}{C(i,s)}\\right), \\\\\n\\mathrm{AuthorGini}(i,s)\n  &= \\mathrm{Gini}\\bigl(\\{C(a,i,s)\\}_a\\bigr).\n\\end{aligne

## 15. Resumo final

| Saída | Definição | Papel na RQ2 |
|---|---|---|
| **M3a** | Participação dos commits em $[P-7d,P+7d)$ sobre commits até $P+7d$ | Quantifica quando a atividade se concentra |
| **M3b** | Concentração de autoria nas fases pré e pós | Descreve quem concentra a atividade final |
| **M3c** | Concentração estrutural, baseline e janelas móveis | Distingue padrão persistente de mudança tardia |

Resultados centrais:

- M3a geral: mediana 43,7%; ponderada 37,9%;
- pico móvel agregado em $P+1d$ nos dois semestres;
- 10/14 equipes com pico individual em $P$ ou $P+1d$;
- concentração estrutural–final: $\rho=0{,}773$ (sobreposição);
- baseline–final: $\rho=0{,}612$ (sem sobreposição);
- nenhuma elevação geral da concentração final.

Variantes mantidas como componentes: `post_share`, momento do pico, concentração pré-final e dias ativos.

Variantes descartadas por redundância: participação da semana de pico e composto do autor dominante. Aceleração semanal permanece exploratória porque satura em 13/14 equipes.

**Recomendação:** M3a como resultado principal; M3b e M3c como explicações complementares. M3 permanece contexto secundário da RQ2, enquanto M4–M5 sustentam o contraste central.